In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q monai

In [ ]:
import monai
import os
import glob
import re
import pandas as pd
import torch
import nibabel as nib
import numpy as np
from monai.transforms import Compose, Resize, NormalizeIntensity

# ============================================================
# PARCHE: convertir cualquier salida de MONAI a torch.Tensor puro
# ============================================================
def to_plain_tensor(x, ensure_channel_dim=True):
    """
    Convierte Tensor / MetaTensor / ndarray a torch.Tensor puro,
    sin metadatos MONAI, en float32 y contiguo.
    """
    if torch.is_tensor(x):
        x = x.detach().clone()
    else:
        x = torch.as_tensor(x)

    x = x.to(torch.float32).contiguous()

    if ensure_channel_dim and x.ndim == 3:
        x = x.unsqueeze(0)   # -> (1, H, W, D)

    if x.ndim != 4:
        raise ValueError(f"Se esperaba tensor con shape (1,H,W,D), llegó {tuple(x.shape)}")

    # importante: reconstruir como Tensor puro para romper con MetaTensor
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)
    return x.contiguous()


def save_plain_mri_pt(x, save_path):
    """
    Guarda SIEMPRE un dict sencillo con tensor puro.
    """
    x_plain = to_plain_tensor(x, ensure_channel_dim=True)
    torch.save({"x": x_plain}, save_path)

In [ ]:
"""
atlas_utils.py
==============
Section C of the Materials and Methods.

Defines an atlas manager that:
  1. Loads an integer-valued atlas in template space.
  2. Extracts ROI masks {R_k}_{k=1}^K.
  3. Resamples those masks either to image space or feature-map space.
  4. Normalizes each ROI mask so that its support sums to 1, matching the
     masked pooling formula used by ROITokenizer.

The module does not assume a specific atlas vendor. It only requires a NIfTI
label map with integer region identifiers.
"""



from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Union

import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

ArrayLikePath = Union[str, Path]


@dataclass
class AtlasConfig:
    label_values: Optional[Sequence[int]] = None
    drop_background: bool = True
    eps: float = 1e-8
    min_voxels_per_roi: int = 1


def load_label_atlas(path: ArrayLikePath):
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)
    atlas = img.get_fdata(dtype=np.float32)

    if atlas.ndim == 4:
        atlas = atlas[..., 0]

    atlas = np.rint(atlas).astype(np.int32)
    return img, atlas


def infer_label_values(
    atlas: np.ndarray,
    drop_background: bool = True,
    min_voxels_per_roi: int = 1,
) -> list[int]:
    vals = np.unique(atlas).tolist()
    vals = [int(v) for v in vals]

    if drop_background:
        vals = [v for v in vals if v != 0]

    if min_voxels_per_roi > 1:
        vals = [v for v in vals if int((atlas == v).sum()) >= min_voxels_per_roi]

    return sorted(vals)


class AtlasROIManager:
    def __init__(self, atlas_path: ArrayLikePath, config: Optional[AtlasConfig] = None):
        self.atlas_path = str(atlas_path)
        self.config = config or AtlasConfig()

        self.atlas_img, self.atlas_np = load_label_atlas(self.atlas_path)
        self.affine = self.atlas_img.affine.copy()
        self.shape = tuple(int(v) for v in self.atlas_np.shape)

        if self.config.label_values is not None:
            self.label_values = [int(v) for v in self.config.label_values]
        else:
            self.label_values = infer_label_values(
                self.atlas_np,
                drop_background=self.config.drop_background,
                min_voxels_per_roi=self.config.min_voxels_per_roi,
            )

        self.K = len(self.label_values)

        self._atlas_onehot = self._build_onehot(self.atlas_np, self.label_values)
        self.atlas_tensor = self._atlas_onehot  # alias público

        self.roi_volumes = self._atlas_onehot.flatten(1).sum(dim=1).long()

        self._validate_nonempty()

    @staticmethod
    def _build_onehot(atlas: np.ndarray, label_values: Sequence[int]) -> torch.Tensor:
        masks = []
        for lab in label_values:
            masks.append((atlas == int(lab)).astype(np.float32))

        if len(masks) == 0:
            raise ValueError("No ROI labels were found in the atlas.")

        onehot = np.stack(masks, axis=0)  # (K, H, W, D)
        return torch.from_numpy(onehot)

    def _validate_nonempty(self):
        empty = (self.roi_volumes <= 0).nonzero(as_tuple=False).flatten().tolist()
        if len(empty) > 0:
            bad_labels = [self.label_values[i] for i in empty]
            raise ValueError(
                f"El atlas contiene ROIs vacías después de cargarlo. "
                f"indices={empty}, labels={bad_labels}"
            )

    @staticmethod
    def _resize_masks(masks: torch.Tensor, target_shape: Sequence[int]) -> torch.Tensor:
        """
        masks: (K, H, W, D)
        output: (K, Ht, Wt, Dt)
        """
        if len(target_shape) != 3:
            raise ValueError(f"target_shape debe tener longitud 3, llegó: {target_shape}")

        x = masks.unsqueeze(1)  # (K,1,H,W,D)
        x = F.interpolate(x, size=tuple(int(v) for v in target_shape), mode="nearest")
        return x.squeeze(1)

    @staticmethod
    def _normalize_masks(masks: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
        flat = masks.flatten(1)
        denom = flat.sum(dim=1, keepdim=True).clamp_min(eps)
        flat = flat / denom
        return flat.view_as(masks)

    def get_masks(
        self,
        target_shape: Sequence[int],
        normalize: bool = True,
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        """
        Devuelve máscaras ROI remuestreadas al shape objetivo.
        Salida: (K, Ht, Wt, Dt)
        """
        masks = self._resize_masks(self._atlas_onehot.float(), target_shape)

        if normalize:
            masks = self._normalize_masks(masks, eps=self.config.eps)

        masks = masks.to(dtype=dtype)

        if device is not None:
            masks = masks.to(device)

        return masks

    def get_binary_masks(
        self,
        target_shape: Sequence[int],
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        masks = self._resize_masks(self._atlas_onehot.float(), target_shape)
        masks = (masks > 0.5).to(dtype=dtype)

        if device is not None:
            masks = masks.to(device)

        return masks

    def roi_weights_from_volume(
        self,
        power: float = 0.0,
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        """
        power = 0.0 -> pesos uniformes
        power > 0.0 -> inverse-volume weighting^power, renormalizado
        """
        vol = self._atlas_onehot.flatten(1).sum(dim=1).float().clamp_min(1.0)

        if power <= 0:
            w = torch.ones_like(vol)
        else:
            w = (1.0 / vol) ** power

        w = w / w.sum().clamp_min(self.config.eps)
        w = w.to(dtype=dtype)

        if device is not None:
            w = w.to(device)

        return w

    def maybe_validate_K(self, K_expected: int) -> None:
        if self.K != int(K_expected):
            raise ValueError(
                f"Atlas has K={self.K} regions, but the model/loss expects K={int(K_expected)}."
            )

    def summary(self) -> dict:
        return {
            "atlas_path": self.atlas_path,
            "shape": self.shape,
            "K": self.K,
            "label_min": int(min(self.label_values)) if self.K > 0 else None,
            "label_max": int(max(self.label_values)) if self.K > 0 else None,
            "n_background_voxels": int((self.atlas_np == 0).sum()),
            "roi_volumes_min": int(self.roi_volumes.min().item()) if self.K > 0 else None,
            "roi_volumes_max": int(self.roi_volumes.max().item()) if self.K > 0 else None,
        }

In [ ]:

"""
concept_targets.py
==================
Section K of the Materials and Methods.

This module implements a practical MRI-only anatomical target c_tilde_{n,k}
for each ROI. Because the Kaggle derivatives may not ship FreeSurfer-based
cortical thickness or per-region morphometric spreadsheets, we implement an
atlas-based tissue-loss summary that is directly computable from the 3D MRI.

The default biomarker is:
    s_{n,k} = fraction of ROI voxels below the subject-specific q-th
              percentile of the intracranial intensity distribution

This makes s_{n,k} a bounded regional tissue-loss proxy. It is then converted
into a concept target in [0,1] through a CN-referenced z-score and a sigmoid:
    c_tilde_{n,k} = sigmoid((s_{n,k} - mu_k_CN) / (sigma_k_CN + eps))

If you later obtain stronger morphometric measurements, only the extractor
function needs to be replaced; the normalizer and cache protocol can remain.
"""

from __future__ import annotations

import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Sequence, Union

import numpy as np
import torch

ArrayLikePath = Union[str, Path]

@dataclass
class ConceptTargetConfig:
    brain_threshold: float = 0.0
    low_intensity_percentile: float = 20.0
    eps: float = 1e-6
    normal_class_name: str = "CN"


def _safe_torch_load(path):
    obj = torch.load(str(path), map_location="cpu", weights_only=False)

    if torch.is_tensor(obj):
        x = obj
    elif isinstance(obj, dict):
        x = None
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                break
        if x is None:
            raise KeyError(f"No se encontró tensor MRI en {path}")
    else:
        x = torch.as_tensor(obj)

    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    x = x.detach().to(torch.float32)
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)
    return x


def _unwrap_tensorlike(obj):
    """
    Accept:
      - plain Tensor / MetaTensor
      - dict with keys like 'x', 'image', 'mri', 'tensor', 'volume'
    """
    if torch.is_tensor(obj):
        return obj

    if isinstance(obj, dict):
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                if torch.is_tensor(x):
                    return x
                return torch.as_tensor(x)

    # final fallback
    return torch.as_tensor(obj)


def _to_numpy_volume(x: torch.Tensor | np.ndarray) -> np.ndarray:
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 4 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 3:
        raise ValueError(f"Expected 3D volume or (1,H,W,D), got shape {x.shape}.")
    return x.astype(np.float32)


def extract_tissue_loss_proxy(
    x: torch.Tensor | np.ndarray,
    atlas_mgr: AtlasROIManager,
    cfg: Optional[ConceptTargetConfig] = None,
) -> np.ndarray:
    cfg = cfg or ConceptTargetConfig()
    vol = _to_numpy_volume(x)
    roi_masks = atlas_mgr.get_binary_masks(vol.shape).cpu().numpy()   # (K,H,W,D)

    brain = vol[vol > cfg.brain_threshold]
    if brain.size == 0:
        raise ValueError("Empty brain mask after thresholding; cannot compute concept targets.")

    q = np.percentile(brain, cfg.low_intensity_percentile)
    proxy = np.zeros(atlas_mgr.K, dtype=np.float32)

    for k in range(atlas_mgr.K):
        mask = roi_masks[k] > 0
        if not np.any(mask):
            continue
        roi_vals = vol[mask]
        proxy[k] = float((roi_vals <= q).mean())

    return proxy.astype(np.float32)



def _to_torch_volume(x, device):
    if torch.is_tensor(x):
        vol = x.detach()
    else:
        vol = torch.as_tensor(x)

    vol = vol.float()

    # admitir (1,H,W,D) o (H,W,D)
    if vol.ndim == 4 and vol.shape[0] == 1:
        vol = vol[0]
    elif vol.ndim != 3:
        raise ValueError(f"Se esperaba volumen 3D o (1,H,W,D), llegó {tuple(vol.shape)}")

    return vol.to(device, non_blocking=True)


@dataclass
class ConceptNormalizer:
    mu: np.ndarray
    sigma: np.ndarray
    eps: float = 1e-6

    def transform(self, features: np.ndarray) -> np.ndarray:
        z = (features - self.mu[None, :]) / (self.sigma[None, :] + self.eps)
        return 1.0 / (1.0 + np.exp(-z))

    def save(self, path: ArrayLikePath) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "mu": self.mu.tolist(),
            "sigma": self.sigma.tolist(),
            "eps": float(self.eps),
        }
        path.write_text(json.dumps(payload, indent=2))

    @classmethod
    def load(cls, path: ArrayLikePath) -> "ConceptNormalizer":
        payload = json.loads(Path(path).read_text())
        return cls(
            mu=np.asarray(payload["mu"], dtype=np.float32),
            sigma=np.asarray(payload["sigma"], dtype=np.float32),
            eps=float(payload["eps"]),
        )


def fit_concept_normalizer(
    features: np.ndarray,
    labels: Sequence[str] | Sequence[int],
    normal_label: str | int = "CN",
    eps: float = 1e-6,
) -> ConceptNormalizer:
    features = np.asarray(features, dtype=np.float32)
    labels = np.asarray(labels)
    mask = labels == normal_label
    if mask.sum() == 0:
        raise ValueError(f"No samples found for normal_label={normal_label!r}.")
    ref = features[mask]
    mu = ref.mean(axis=0).astype(np.float32)
    sigma = ref.std(axis=0).astype(np.float32)
    return ConceptNormalizer(mu=mu, sigma=sigma, eps=eps)



def build_subject_concept_target(
    x: torch.Tensor | np.ndarray,
    atlas_mgr: AtlasROIManager,
    normalizer: ConceptNormalizer,
    cfg: Optional[ConceptTargetConfig] = None,
) -> torch.Tensor:
    feats = extract_tissue_loss_proxy(x, atlas_mgr, cfg=cfg)[None, :]
    c_tilde = normalizer.transform(feats)[0]
    return torch.from_numpy(c_tilde.astype(np.float32))


def precompute_concept_targets_from_dataframe(
    df,
    atlas_mgr: AtlasROIManager,
    x_column: str = "x_path",
    label_column: str = "label",
    subject_id_column: str = "subject_id",
    output_dir: ArrayLikePath = "./concept_targets",
    cfg: Optional[ConceptTargetConfig] = None,
) -> tuple[ConceptNormalizer, "pd.DataFrame"]:
    import pandas as pd

    cfg = cfg or ConceptTargetConfig()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_features = []
    labels = []
    rows = []

    for _, row in df.iterrows():
        x = _safe_torch_load(row[x_column])
        feats = extract_tissue_loss_proxy(x, atlas_mgr, cfg=cfg)
        raw_features.append(feats)
        labels.append(row[label_column])

        rows.append({
            "subject_id": row[subject_id_column],
            "label": row[label_column],
            "x_path": row[x_column],
        })

    raw_features = np.stack(raw_features, axis=0)
    normalizer = fit_concept_normalizer(
        raw_features,
        labels=labels,
        normal_label=cfg.normal_class_name,
        eps=cfg.eps,
    )

    out_rows = []
    transformed = normalizer.transform(raw_features)

    for meta, c_tilde in zip(rows, transformed):
        save_path = output_dir / f"{meta['subject_id']}_c_target.pt"
        save_plain_vector_pt(torch.from_numpy(c_tilde.astype(np.float32)), save_path, key="c_target")

        out_rows.append({
            **meta,
            "concept_target_path": str(save_path),
        })

    normalizer.save(output_dir / "concept_normalizer.json")
    return normalizer, pd.DataFrame(out_rows)

In [ ]:

"""
jacobian_utils.py
=================
Section L of the Materials and Methods.

This module computes:
    g_{n,k}   = mean_{x in R_k} psi(J_n(x))
    g_bar     = normalized ROI-wise deformation summary

It supports two regimes:
  1. If a displacement field already exists, compute Jacobian directly.
  2. If only template and subject MRI are available, estimate a non-linear
     displacement field with SimpleITK (Demons registration).

The implementation is deliberately explicit because Jacobian-based terms are
part of the anatomical plausibility regularizer, not pathology ground truth.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Union

import nibabel as nib
import numpy as np
import torch

# from atlas_utils import AtlasROIManager

try:
    import SimpleITK as sitk
except Exception:  # pragma: no cover
    sitk = None

ArrayLikePath = Union[str, Path]

@dataclass
class JacobianConfig:
    psi: str = "neg_log"
    eps: float = 1e-6
    n_iterations: int = 50
    smooth_displacement_field: bool = True
    normalize_within_subject: bool = True


def _safe_torch_load(path):
    """
    Carga .pt confiables del pipeline propio.
    Soporta archivos antiguos que contienen MONAI MetaTensor.
    """
    obj = torch.load(str(path), map_location="cpu", weights_only=False)

    # Caso 1: tensor directo / MetaTensor
    if torch.is_tensor(obj):
        x = obj

    # Caso 2: dict con tensor MRI
    elif isinstance(obj, dict):
        x = None
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                break
        if x is None:
            raise KeyError(f"No se encontró tensor MRI en {path}")
    else:
        x = torch.as_tensor(obj)

    # romper dependencia con MetaTensor
    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    x = x.detach().to(torch.float32)
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)  # fuerza Tensor puro
    return x

def _unwrap_tensorlike(obj):
    if torch.is_tensor(obj):
        return obj

    if isinstance(obj, dict):
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                if torch.is_tensor(x):
                    return x.detach().to(torch.float32)
                return torch.as_tensor(x, dtype=torch.float32)

    raise TypeError(f"Unsupported object type loaded from checkpoint: {type(obj)}")


def _tensor_to_3d_numpy(x) -> np.ndarray:
    if torch.is_tensor(x):
        x = x.detach().cpu().to(torch.float32)
        if x.ndim == 4 and x.shape[0] == 1:
            x = x[0]
        arr = x.numpy()
    else:
        arr = np.asarray(x, dtype=np.float32)
        if arr.ndim == 4 and arr.shape[0] == 1:
            arr = arr[0]

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume or (1,H,W,D), got shape {arr.shape}")

    return np.ascontiguousarray(arr.astype(np.float32))


def _ensure_sitk():
    if sitk is None:
        raise ImportError(
            "SimpleITK is required for Jacobian computation from displacement fields "
            "or for Demons registration. Install SimpleITK before using jacobian_utils.py."
        )

def load_nifti_array(path: ArrayLikePath) -> np.ndarray:
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)
    arr = img.get_fdata(dtype=np.float32)
    if arr.ndim == 4:
        arr = arr[..., 0]
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def sitk_from_numpy(arr: np.ndarray) -> "sitk.Image":
    _ensure_sitk()
    img = sitk.GetImageFromArray(arr.astype(np.float32))
    img.SetSpacing((1.0, 1.0, 1.0))
    return img

def estimate_displacement_field(
    fixed_volume: np.ndarray,
    moving_volume: np.ndarray,
    cfg: Optional[JacobianConfig] = None,
) -> "sitk.Image":
    _ensure_sitk()
    cfg = cfg or JacobianConfig()

    fixed = sitk_from_numpy(fixed_volume)
    moving = sitk_from_numpy(moving_volume)

    matcher = sitk.HistogramMatchingImageFilter()
    matcher.SetNumberOfHistogramLevels(128)
    matcher.SetNumberOfMatchPoints(10)
    moving = matcher.Execute(moving, fixed)

    demons = sitk.DiffeomorphicDemonsRegistrationFilter()
    demons.SetNumberOfIterations(int(cfg.n_iterations))
    demons.SetStandardDeviations(1.0)
    displacement = demons.Execute(fixed, moving)

    if cfg.smooth_displacement_field:
        displacement = sitk.SmoothingRecursiveGaussian(displacement, 1.0)
    return displacement

def jacobian_determinant_from_displacement(displacement_field: "sitk.Image") -> np.ndarray:
    _ensure_sitk()
    jac = sitk.DisplacementFieldJacobianDeterminant(displacement_field)
    jac_np = sitk.GetArrayFromImage(jac).astype(np.float32)
    return np.nan_to_num(jac_np, nan=1.0, posinf=1.0, neginf=1.0)

def apply_psi(jac_det: np.ndarray, psi: str = "neg_log", eps: float = 1e-6) -> np.ndarray:
    jac_det = np.clip(jac_det, eps, None)
    if psi == "neg_log":
        return -np.log(jac_det).astype(np.float32)
    if psi == "identity":
        return jac_det.astype(np.float32)
    raise ValueError(f"Unknown psi={psi!r}")

def pool_roi_deformation(
    psi_jacobian: np.ndarray,
    atlas_mgr: AtlasROIManager,
) -> np.ndarray:
    masks = atlas_mgr.get_binary_masks(psi_jacobian.shape).cpu().numpy()
    out = np.zeros(atlas_mgr.K, dtype=np.float32)

    for k in range(atlas_mgr.K):
        mask = masks[k] > 0
        if np.any(mask):
            out[k] = float(psi_jacobian[mask].mean())
    return out

def normalize_roi_summary(g: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    g = np.asarray(g, dtype=np.float32)
    mu = float(g.mean())
    sigma = float(g.std())
    z = (g - mu) / (sigma + eps)
    return (1.0 / (1.0 + np.exp(-z))).astype(np.float32)

def compute_g_bar_from_template_and_subject(
    template_volume: np.ndarray,
    subject_volume: np.ndarray,
    atlas_mgr: AtlasROIManager,
    cfg: Optional[JacobianConfig] = None,
) -> torch.Tensor:
    cfg = cfg or JacobianConfig()
    displacement = estimate_displacement_field(template_volume, subject_volume, cfg=cfg)
    jac_det = jacobian_determinant_from_displacement(displacement)
    psi_jac = apply_psi(jac_det, psi=cfg.psi, eps=cfg.eps)
    g = pool_roi_deformation(psi_jac, atlas_mgr)

    if cfg.normalize_within_subject:
        g = normalize_roi_summary(g, eps=cfg.eps)
    return torch.from_numpy(g.astype(np.float32))

def to_plain_vector(x, expected_dim=1):
    """
    Convierte un vector ROI (por ejemplo g_bar o c_target) a torch.Tensor puro.
    """
    if torch.is_tensor(x):
        x = x.detach().clone()
    else:
        x = torch.as_tensor(x)

    x = x.to(torch.float32).contiguous()

    if expected_dim is not None and x.ndim != expected_dim:
        raise ValueError(f"Se esperaba tensor con ndim={expected_dim}, llegó shape={tuple(x.shape)}")

    # romper dependencia con MetaTensor si existiera
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)
    return x.contiguous()


def save_plain_vector_pt(x, save_path, key="x"):
    """
    Guarda un vector anatómico puro, por ejemplo g_bar o c_target.
    """
    x_plain = to_plain_vector(x, expected_dim=1)
    torch.save({key: x_plain}, save_path)

def precompute_jacobians_from_dataframe(
    df,
    atlas_mgr: AtlasROIManager,
    template_x_path: ArrayLikePath,
    x_column: str = "x_path",
    subject_id_column: str = "subject_id",
    output_dir: ArrayLikePath = "./jacobian_targets",
    cfg: Optional[JacobianConfig] = None,
):
    import pandas as pd

    cfg = cfg or JacobianConfig()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    template_obj = _safe_torch_load(template_x_path)
    template_x = _tensor_to_3d_numpy(_safe_torch_load(template_x_path))

    rows = []
    for _, row in df.iterrows():
        x_obj = _safe_torch_load(row[x_column])
        x = _tensor_to_3d_numpy(_safe_torch_load(row[x_column]))

        g_bar = compute_g_bar_from_template_and_subject(
            template_volume=template_x,
            subject_volume=x,
            atlas_mgr=atlas_mgr,
            cfg=cfg,
        )

        save_path = output_dir / f"{row[subject_id_column]}_g_bar.pt"
        # torch.save(g_bar, save_path)
        save_plain_vector_pt(g_bar, save_path, key="g_bar")

        rows.append({
            "subject_id": row[subject_id_column],
            "x_path": row[x_column],
            "g_bar_path": str(save_path),
        })

    return pd.DataFrame(rows)

In [ ]:
import os
import glob
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 1. LABEL ENCODING MAPPING
# =====================================================================
LABEL_MAP = {
    'CN': 0,
    'MCI': 1,
    'AD': 2
}

def ensure_artifact_cache(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    recompute: bool = False,
    cache_dir: Optional[str] = None,
) -> dict:
    add_module_dir_to_path(module_dir)

    from atlas_utils import AtlasROIManager
    from concept_targets import ConceptTargetConfig, precompute_concept_targets_from_dataframe
    from jacobian_utils import JacobianConfig, precompute_jacobians_from_dataframe

    df_source, df_target = build_inventory_dataframes(base_dir)

    if cache_dir is None:
        cache_dir = _default_cache_dir(base_dir)

    artifacts_dir = cache_dir
    src_concepts_dir = os.path.join(artifacts_dir, "source_concepts")
    src_jac_dir = os.path.join(artifacts_dir, "source_jacobians")
    tgt_jac_dir = os.path.join(artifacts_dir, "target_jacobians")
    os.makedirs(src_concepts_dir, exist_ok=True)
    os.makedirs(src_jac_dir, exist_ok=True)
    os.makedirs(tgt_jac_dir, exist_ok=True)

    atlas_mgr = AtlasROIManager(atlas_path)
    template_x_path = choose_template_x_path(df_source)

    source_inventory_csv = os.path.join(artifacts_dir, "source_inventory.csv")
    target_inventory_csv = os.path.join(artifacts_dir, "target_inventory.csv")
    source_concepts_csv = os.path.join(artifacts_dir, "source_concepts_index.csv")
    source_jac_csv = os.path.join(artifacts_dir, "source_jacobians_index.csv")
    target_jac_csv = os.path.join(artifacts_dir, "target_jacobians_index.csv")
    cache_meta_json = os.path.join(artifacts_dir, "cache_meta.json")

    df_source.to_csv(source_inventory_csv, index=False)
    df_target.to_csv(target_inventory_csv, index=False)

    if recompute or not os.path.exists(source_concepts_csv):
        _, df_concepts = precompute_concept_targets_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            x_column="x_path",
            label_column="label",
            subject_id_column="subject_id",
            output_dir=src_concepts_dir,
            cfg=ConceptTargetConfig(normal_class_name="CN"),
        )
        df_concepts.to_csv(source_concepts_csv, index=False)
    else:
        df_concepts = pd.read_csv(source_concepts_csv)

    if recompute or not os.path.exists(source_jac_csv):
        ensure_simpleitk_or_raise()
        df_src_jac = precompute_jacobians_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=src_jac_dir,
            cfg=JacobianConfig(),
        )
        df_src_jac.to_csv(source_jac_csv, index=False)
    else:
        df_src_jac = pd.read_csv(source_jac_csv)

    if recompute or not os.path.exists(target_jac_csv):
        ensure_simpleitk_or_raise()
        df_tgt_jac = precompute_jacobians_from_dataframe(
            df=df_target,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=tgt_jac_dir,
            cfg=JacobianConfig(),
        )
        df_tgt_jac.to_csv(target_jac_csv, index=False)
    else:
        df_tgt_jac = pd.read_csv(target_jac_csv)

    meta = {
        "base_dir": base_dir,
        "cache_dir": artifacts_dir,
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": int(atlas_mgr.K),
        "n_source": int(len(df_source)),
        "n_target": int(len(df_target)),
    }
    with open(cache_meta_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return {
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": atlas_mgr.K,
        "df_source": df_source,
        "df_target": df_target,
        "df_concepts": df_concepts,
        "df_src_jac": df_src_jac,
        "df_tgt_jac": df_tgt_jac,
        "source_inventory_csv": source_inventory_csv,
        "target_inventory_csv": target_inventory_csv,
        "source_concepts_csv": source_concepts_csv,
        "source_jac_csv": source_jac_csv,
        "target_jac_csv": target_jac_csv,
        "cache_dir": artifacts_dir,
        "cache_meta_json": cache_meta_json,
    }

def _resolve_artifact_path(old_path: str, artifacts_dir: str, subdir: str) -> str:
    """
    Si el path serializado ya no existe (porque viene de /kaggle/working del notebook
    anterior), lo remapea al dataset montado actual dentro de precomputed_artifacts_dir.
    """
    old_path = str(old_path)

    if os.path.exists(old_path):
        return old_path

    fname = os.path.basename(old_path)

    candidates = [
        os.path.join(artifacts_dir, subdir, fname),
        os.path.join(artifacts_dir, fname),
    ]

    for c in candidates:
        if os.path.exists(c):
            return c

    matches = glob.glob(os.path.join(artifacts_dir, "**", fname), recursive=True)
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Artifact file not found.\n"
        f"Original serialized path: {old_path}\n"
        f"Searched under: {artifacts_dir}"
    )


def _remap_artifact_dataframe_paths(df: pd.DataFrame, col: str, artifacts_dir: str, subdir: str) -> pd.DataFrame:
    df = df.copy()
    df[col] = df[col].apply(lambda p: _resolve_artifact_path(p, artifacts_dir, subdir))
    return df


import os
import glob
import pandas as pd

def _resolve_artifact_path(old_path: str, artifacts_dir: str, subdir: str) -> str:
    """
    Remapea rutas serializadas antiguas de /kaggle/working al dataset montado actual.
    """
    old_path = str(old_path)

    # caso ideal: la ruta serializada todavía existe
    if os.path.exists(old_path):
        return old_path

    fname = os.path.basename(old_path)

    # búsqueda determinista primero
    candidates = [
        os.path.join(artifacts_dir, subdir, fname),
        os.path.join(artifacts_dir, fname),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c

    # búsqueda recursiva final
    matches = glob.glob(os.path.join(artifacts_dir, "**", fname), recursive=True)
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"No se encontró el artefacto.\n"
        f"Ruta serializada original: {old_path}\n"
        f"Directorio de artefactos actual: {artifacts_dir}"
    )


def _remap_artifact_dataframe_paths(df: pd.DataFrame, col: str, artifacts_dir: str, subdir: str) -> pd.DataFrame:
    df = df.copy()
    df[col] = df[col].apply(lambda p: _resolve_artifact_path(p, artifacts_dir, subdir))
    return df

def load_precomputed_artifacts(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
) -> dict:
    add_module_dir_to_path(module_dir)

    artifacts_dir = str(precomputed_artifacts_dir)
    if not os.path.isdir(artifacts_dir):
        raise FileNotFoundError(f"precomputed_artifacts_dir no existe: {artifacts_dir}")

    source_concepts_csv = os.path.join(artifacts_dir, "source_concepts_index.csv")
    source_jac_csv      = os.path.join(artifacts_dir, "source_jacobians_index.csv")
    target_jac_csv      = os.path.join(artifacts_dir, "target_jacobians_index.csv")
    target_concepts_csv = os.path.join(artifacts_dir, "target_concepts_index.csv")

    source_inventory_csv = os.path.join(artifacts_dir, "source_inventory.csv")
    target_inventory_csv = os.path.join(artifacts_dir, "target_inventory.csv")

    for p in [source_concepts_csv, source_jac_csv, target_jac_csv]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Falta archivo requerido: {p}")

    if os.path.exists(source_inventory_csv) and os.path.exists(target_inventory_csv):
        df_source = pd.read_csv(source_inventory_csv)
        df_target = pd.read_csv(target_inventory_csv)
    else:
        df_source, df_target = build_inventory_dataframes(base_dir)

    df_concepts = pd.read_csv(source_concepts_csv)
    df_src_jac  = pd.read_csv(source_jac_csv)
    df_tgt_jac  = pd.read_csv(target_jac_csv)

    df_tgt_concepts = None
    if os.path.exists(target_concepts_csv):
        df_tgt_concepts = pd.read_csv(target_concepts_csv)

    # remapeo de rutas antiguas serializadas
    df_concepts = _remap_artifact_dataframe_paths(
        df_concepts, "concept_target_path", artifacts_dir, "source_concepts"
    )
    df_src_jac = _remap_artifact_dataframe_paths(
        df_src_jac, "g_bar_path", artifacts_dir, "source_jacobians"
    )
    df_tgt_jac = _remap_artifact_dataframe_paths(
        df_tgt_jac, "g_bar_path", artifacts_dir, "target_jacobians"
    )

    if df_tgt_concepts is not None:
        df_tgt_concepts = _remap_artifact_dataframe_paths(
            df_tgt_concepts, "concept_target_path", artifacts_dir, "target_concepts"
        )

    atlas_mgr = AtlasROIManager(atlas_path)
    template_x_path = choose_template_x_path(df_source)

    return {
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": int(atlas_mgr.K),
        "df_source": df_source,
        "df_target": df_target,
        "df_concepts": df_concepts,
        "df_src_jac": df_src_jac,
        "df_tgt_jac": df_tgt_jac,
        "df_tgt_concepts": df_tgt_concepts,
        "cache_dir": artifacts_dir,
    }

def _index_by_subject(df: pd.DataFrame, path_col: str) -> Dict[str, str]:
    out = {}
    for _, row in df.iterrows():
        out[str(row["subject_id"])] = str(row[path_col])
    return out


def _load_vector(path: str, expected_last_dim: int) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu", weights_only=False)

    if torch.is_tensor(obj):
        v = obj

    elif isinstance(obj, dict):
        v = None
        for key in ["c_target", "g_bar", "x", "tensor", "vector"]:
            if key in obj:
                v = obj[key]
                break
        if v is None:
            raise KeyError(
                f"No se encontró ningún vector válido en {path}. "
                f"Keys disponibles: {list(obj.keys())}"
            )
    else:
        raise TypeError(f"Unsupported object at {path}: {type(obj)}")

    if not torch.is_tensor(v):
        v = torch.as_tensor(v)

    v = v.detach().to(torch.float32).view(-1)

    if v.numel() != expected_last_dim:
        raise ValueError(
            f"Expected vector with K={expected_last_dim} at {path}, got shape {tuple(v.shape)}"
        )
    # romper cualquier posible dependencia residual con MetaTensor
    v = torch.tensor(v.cpu().numpy(), dtype=torch.float32)
    return v


# ---------------------------------------------------------------------
# 4) ARTIFACT CACHE CREATION (K and L)
# ---------------------------------------------------------------------
def find_existing_atlas_path(explicit_atlas_path: Optional[str] = None) -> str:
    if explicit_atlas_path is not None and os.path.exists(explicit_atlas_path):
        return explicit_atlas_path

    patterns = [
        "/kaggle/input/**/*atlas*.nii*",
        "/kaggle/input/**/*aal*.nii*",
        "/kaggle/input/**/*harvard*oxford*.nii*",
        "/kaggle/input/**/*label*.nii*",
        "/kaggle/working/**/*atlas*.nii*",
    ]
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    hits = sorted(set(hits))

    if not hits:
        raise FileNotFoundError(
            "Atlas file not found automatically. Please pass atlas_path explicitly."
        )
    return hits[0]


def ensure_simpleitk_or_raise():
    try:
        import SimpleITK  # noqa: F401
    except Exception as e:
        raise ImportError(
            "SimpleITK is required to compute g_bar Jacobian priors. "
            "Install it in Kaggle before running artifact generation."
        ) from e


def choose_template_x_path(df_source: pd.DataFrame) -> str:
    cn = df_source[df_source["label"] == "CN"].reset_index(drop=True)
    if len(cn) == 0:
        # fallback: first source sample
        return str(df_source.iloc[0]["x_path"])
    return str(cn.iloc[0]["x_path"])

# =====================================================================
# 2. SOURCE DOMAIN DATASET (ADNI)
# =====================================================================
class SourceDomainDataset(Dataset):
    def __init__(self, csv_path, source_artifacts_dir):
        super().__init__()
        self.source_artifacts_dir = source_artifacts_dir
        
        df = pd.read_csv(csv_path)
        self.data = df[df['Label'].isin(LABEL_MAP.keys())].reset_index(drop=True)
        
        # [FIX] DYNAMIC PATH RESOLVER FOR ADNI
        # Instead of trusting the CSV path, we find exactly where the ADNI dataset 
        # is currently mounted in this specific Kaggle session to prevent FileNotFoundError.
        print("Locating ADNI source tensors dynamically...")
        self.mri_path_map = {}
        # Search Kaggle input for all ADNI .pt files
        adni_files = glob.glob("/kaggle/input/**/*.pt", recursive=True)
        for f in adni_files:
            # Only map the actual MRI tensors, ignore concepts/jacobians and oasis
            if "c_target" not in f and "g_jacobian" not in f and "oasis" not in f:
                basename = os.path.basename(f).replace('.pt', '')
                self.mri_path_map[basename] = f
                
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row['Subject_ID'])
        label_str = row['Label']
        
        # 1. Load Preprocessed MRI Tensor (X^S)
        # Look up the newly resolved path from our dictionary
        if sub_id in self.mri_path_map:
            mri_path = self.mri_path_map[sub_id]
        else:
            # Fallback if mapping failed (relies on standard Kaggle structure)
            mri_path = str(row['File_Path']).replace("/datasets/sanjayjoshy/", "/")
            
        # Safely load to CPU first (avoids CUDA initialization errors in workers)
        image_tensor = torch.load(mri_path, map_location='cpu', weights_only=False)
        
        if image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
            
        # 2. Extract Label (y^S)
        label_tensor = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        
        # 3. Load Concept Target & Jacobian Prior (from the new mounted directory)
        c_target = torch.load(os.path.join(self.source_artifacts_dir, f"{sub_id}_c_target.pt"), map_location='cpu', weights_only=False)
        g_jacobian = torch.load(os.path.join(self.source_artifacts_dir, f"{sub_id}_g_jacobian.pt"), map_location='cpu', weights_only=False)
        
        return image_tensor, label_tensor, c_target, g_jacobian

# =====================================================================
# 3. TARGET DOMAIN DATASET (OASIS-1)
# =====================================================================
class TargetDomainDataset(Dataset):
    def __init__(self, csv_path, base_dir):
        super().__init__()
        self.base_dir = base_dir
        
        df = pd.read_csv(csv_path)
        self.data = df[df['Label'].isin(LABEL_MAP.keys())].reset_index(drop=True)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row['Subject_ID'])
        label_str = row['Label']
        
        # [FIX] DYNAMIC TARGET PATH RECONSTRUCTION
        # Completely ignores the old /kaggle/working/ path in the CSV
        mri_path = os.path.join(self.base_dir, "target_oasis", label_str, f"{sub_id}_MRI.pt")
        
        image_tensor = torch.load(mri_path, map_location='cpu', weights_only=False)
        if image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
            
        label_tensor = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        return image_tensor, label_tensor

# =====================================================================
# 4. DATALOADER INITIALIZATION WRAPPER
# =====================================================================
def get_domain_adaptation_dataloaders(
    base_dir="/kaggle/input/notebooks/alejopatio/preprocess-alzheimer/model_ready_data", 
    batch_size=4
):
    source_csv = os.path.join(base_dir, "source_labels.csv")
    target_csv = os.path.join(base_dir, "target_labels.csv")
    source_artifacts_dir = os.path.join(base_dir, "source_adni")
    
    source_dataset = SourceDomainDataset(csv_path=source_csv, source_artifacts_dir=source_artifacts_dir)
    target_dataset = TargetDomainDataset(csv_path=target_csv, base_dir=base_dir)
    
    print(f"Loaded Source Domain (ADNI): {len(source_dataset)} subjects.")
    print(f"Loaded Target Domain (OASIS): {len(target_dataset)} subjects.")
    
    # [FIX] Automatically disables pin_memory if no GPU is found
    use_pin_memory = torch.cuda.is_available()
    
    source_loader = DataLoader(
        source_dataset, batch_size=batch_size, shuffle=True, drop_last=True, 
        num_workers=2, pin_memory=use_pin_memory
    )
    
    target_loader = DataLoader(
        target_dataset, batch_size=batch_size, shuffle=True, drop_last=True, 
        num_workers=2, pin_memory=use_pin_memory
    )
    
    return source_loader, target_loader


In [ ]:
"""
model.py
========
MRI-only, source-target domain adaptation model for Alzheimer diagnosis.

Architecture pipeline (Section D → Q of the M&M):

  X̄  ──► E_θ ──► F          3D hierarchical encoder          (Sec. D)
          │
          ▼
          ROI Tokenizer       masked pooling + projection       (Sec. E)
          │
          ▼
  T  ──► Ψ ──► U             contextual ROI encoder (Transformer) (Sec. F)
                │
                ▼
          Attention Aggregation ──► z                           (Sec. G)
                │
          ┌─────┴───────┐
          ▼             ▼
   Concept Head       Class Head
   c ∈ ℝᴷ           p ∈ Δᶜ                                   (Sec. H, J)
          │
          ▼
   CBM Classifier
   p̃ ∈ Δᶜ                                                    (Sec. J)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


# ---------------------------------------------------------------------------
# A.  3D Hierarchical CNN Encoder  (Section D)
# ---------------------------------------------------------------------------

class ResBlock3D(nn.Module):
    """Basic 3D residual block with GroupNorm."""

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        groups = min(8, out_ch)
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(groups, out_ch)

        self.skip = (
            nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.GroupNorm(groups, out_ch),
            )
            if (in_ch != out_ch or stride != 1)
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.gn1(self.conv1(x)), inplace=True)
        out = self.gn2(self.conv2(out))
        return F.relu(out + self.skip(x), inplace=True)


class Encoder3D(nn.Module):
    """
    Hierarchical 3D CNN encoder (Section D).

    Input  : X̄ ∈ ℝ^{H×W×D}  (single-channel MRI, add batch + channel dims)
    Output : F ∈ ℝ^{h×w×d×C_f}

    Default channel progression: 1 → 32 → 64 → 128 → C_f
    with stride-2 downsampling at each stage (so h=H/8, w=W/8, d=D/8).
    """

    def __init__(self, C_f: int = 256, base_ch: int = 32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(1, base_ch, kernel_size=7, stride=2, padding=3, bias=False),
            nn.GroupNorm(min(8, base_ch), base_ch),
            nn.ReLU(inplace=True),
        )

        self.layer1 = self._make_layer(base_ch,      base_ch * 2,  stride=2)
        self.layer2 = self._make_layer(base_ch * 2,  base_ch * 4,  stride=2)
        self.layer3 = self._make_layer(base_ch * 4,  C_f,          stride=1)
        self.C_f = C_f

    @staticmethod
    def _make_layer(in_ch: int, out_ch: int, stride: int = 1) -> nn.Sequential:
        return nn.Sequential(
            ResBlock3D(in_ch, out_ch, stride=stride),
            ResBlock3D(out_ch, out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (B, 1, H, W, D)
        Returns:
            F : (B, C_f, h, w, d)
        """
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x                          # (B, C_f, h, w, d)


# ---------------------------------------------------------------------------
# B.  ROI Tokenizer  (Section E)
# ---------------------------------------------------------------------------

class ROITokenizer(nn.Module):
    """
    Converts an encoded feature map F into a token matrix T (Section E).

    For each ROI R_k, computes a masked average-pool of F, then projects it:
        r_{n,k}  = mean_{u ∈ R_k}  F_n(u)          ∈ ℝ^{C_f}
        t_{n,k}  = W_r · r_{n,k} + b_r + e_k       ∈ ℝ^{C_t}

    Args:
        K    : number of ROIs
        C_f  : feature channels from encoder
        C_t  : token dimension
    """

    def __init__(self, K: int, C_f: int, C_t: int):
        super().__init__()
        self.K   = K
        self.C_f = C_f
        self.C_t = C_t

        self.proj    = nn.Linear(C_f, C_t)           # W_r, b_r
        self.roi_emb = nn.Embedding(K, C_t)          # e_k

    def forward(
        self,
        F: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            F         : (B, C_f, h, w, d)
            roi_masks : (K, h, w, d)  binary float masks, already resampled to
                        feature-map resolution and normalised so each mask sums
                        to 1 over its support (i.e. divide by |R_k| offline).

        Returns:
            T : (B, K, C_t)
        """
        B = F.shape[0]
        # F_flat : (B, C_f, h*w*d)
        F_flat = F.view(B, self.C_f, -1)

        # roi_masks_flat : (K, h*w*d)
        m_flat = roi_masks.view(self.K, -1)           # (K, N_vox)

        # r_{n,k} = F_flat @ m_flat^T  →  (B, K, C_f)
        R = torch.einsum("bcv,kv->bkc", F_flat, m_flat)

        # Project to token space
        T = self.proj(R)                              # (B, K, C_t)

        # Add learnable ROI position embeddings
        k_idx = torch.arange(self.K, device=F.device)
        T = T + self.roi_emb(k_idx).unsqueeze(0)     # broadcast over B

        return T                                      # (B, K, C_t)


# ---------------------------------------------------------------------------
# C.  Contextual ROI Encoder  Ψ  (Section F)
# ---------------------------------------------------------------------------

class ContextualROIEncoder(nn.Module):
    """
    Shallow Transformer encoder over ROI tokens (Section F).

    Models inter-regional dependencies (hippocampus ↔ entorhinal, etc.)
    via multi-head self-attention with residual connections.

    Args:
        C_t      : token dimension
        n_heads  : number of attention heads
        n_layers : number of Transformer layers
        ffn_mult : hidden-dim multiplier for the FFN
        dropout  : dropout probability
    """

    def __init__(
        self,
        C_t: int,
        n_heads: int = 4,
        n_layers: int = 2,
        ffn_mult: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=C_t,
            nhead=n_heads,
            dim_feedforward=C_t * ffn_mult,
            dropout=dropout,
            batch_first=True,
            norm_first=True,          # pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, T: torch.Tensor) -> torch.Tensor:
        """
        Args:
            T : (B, K, C_t)
        Returns:
            U : (B, K, C_t)
        """
        return self.transformer(T)    # (B, K, C_t)


# ---------------------------------------------------------------------------
# D.  Attention-based Global Aggregation  (Section G)
# ---------------------------------------------------------------------------

class AttentionAggregator(nn.Module):
    """
    Computes a subject-level embedding z by soft attention over ROI tokens U.

        a_{n,k}   = v^T tanh(W_a u_{n,k} + b_a)
        α_{n,k}   = softmax_k(a_{n,k})
        z_n       = Σ_k α_{n,k} u_{n,k}       ∈ ℝ^{C_t}

    Args:
        C_t : token / embedding dimension
    """

    def __init__(self, C_t: int):
        super().__init__()
        self.W_a = nn.Linear(C_t, C_t, bias=True)
        self.v   = nn.Linear(C_t, 1,   bias=False)

    def forward(self, U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            U : (B, K, C_t)
        Returns:
            z     : (B, C_t)
            alpha : (B, K)   attention weights (stored for interpretability)
        """
        a     = self.v(torch.tanh(self.W_a(U))).squeeze(-1)   # (B, K)
        alpha = torch.softmax(a, dim=-1)                        # (B, K)
        z     = (alpha.unsqueeze(-1) * U).sum(dim=1)           # (B, C_t)
        return z, alpha


# ---------------------------------------------------------------------------
# E.  Classification Head  (Section H)
# ---------------------------------------------------------------------------

class ClassificationHead(nn.Module):
    """
    Linear classifier on global embedding z (Section H).

        p_n = softmax(W_c z_n + b_c)

    Returns logits (softmax is applied in the loss).
    """

    def __init__(self, C_t: int, n_classes: int):
        super().__init__()
        self.fc = nn.Linear(C_t, n_classes)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z      : (B, C_t)
        Returns:
            logits : (B, C)
        """
        return self.fc(z)


# ---------------------------------------------------------------------------
# F.  Concept Bottleneck  (Sections J–K)
# ---------------------------------------------------------------------------

class ConceptBottleneck(nn.Module):
    """
    Per-ROI concept predictor and concept-based classifier (Section J).

    For each ROI k and subject n:
        c_{n,k} = σ(w_k^T u_{n,k} + b_k)     scalar ∈ [0,1]

    Then:
        p̃_n = softmax(W_cbm c_n + b_cbm)

    Args:
        K          : number of ROIs / concepts
        C_t        : token dimension
        n_classes  : number of diagnostic classes
    """

    # def __init__(self, K: int, C_t: int, n_classes: int):
    #     super().__init__()
    #     self.K = K
    #     self.C_t = C_t

    #     self.concept_weights = nn.Parameter(torch.empty(K, C_t))
    #     self.concept_bias = nn.Parameter(torch.zeros(K))
    #     nn.init.xavier_uniform_(self.concept_weights)

    #     self.cbm_head = nn.Linear(K, n_classes)

    # def forward(self, U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    #     # U: (B, K, C_t)
    #     raw = torch.einsum("bkc,kc->bk", U, self.concept_weights) + self.concept_bias
    #     c = torch.sigmoid(raw)
    #     cbm_logits = self.cbm_head(c)
    #     return c, cbm_logits

    def __init__(self, K: int, C_t: int, n_classes: int, hidden: int = 64, p: float = 0.2):
        super().__init__()
        self.K = K
        self.concept_mlps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(C_t, hidden),
                nn.GELU(),
                nn.Dropout(p),
                nn.Linear(hidden, 1)
            )
            for _ in range(K)
        ])
        self.cbm_head = nn.Linear(K, n_classes)

    def forward(self, U: torch.Tensor):
        c_list = []
        for k in range(self.K):
            ck = self.concept_mlps[k](U[:, k, :])   # (B,1)
            c_list.append(ck)
        raw = torch.cat(c_list, dim=1)              # (B,K)
        c = torch.sigmoid(raw)
        cbm_logits = self.cbm_head(c)
        return c, cbm_logits
    



# ---------------------------------------------------------------------------
# G.  Full Model  (Sections D–Q)
# ---------------------------------------------------------------------------


class AlzheimerDomainAdaptationModel(nn.Module):
    """
    Unified MRI-only model for Alzheimer diagnosis with domain adaptation.

    Forward pass returns a ModelOutput dataclass containing all intermediate
    tensors needed by the compound loss in losses.py.

    Args:
        K          : number of atlas ROIs
        C_f        : encoder output channels
        C_t        : token / embedding dimension
        n_classes  : number of diagnostic classes (C)
        n_heads    : Transformer attention heads
        n_layers   : Transformer layers
        base_ch    : base channel width in the 3D encoder stem
    """

    def __init__(
        self,
        K:          int = 84,
        C_f:        int = 256,
        C_t:        int = 128,
        n_classes:  int = 3,
        n_heads:    int = 4,
        n_layers:   int = 2,
        base_ch:    int = 32,
    ):
        super().__init__()
        self.K         = K
        self.C_f       = C_f
        self.C_t       = C_t
        self.n_classes = n_classes

        # --- modules (one per M&M section) ---
        self.encoder    = Encoder3D(C_f=C_f, base_ch=base_ch)
        self.tokenizer  = ROITokenizer(K=K, C_f=C_f, C_t=C_t)
        self.token_norm = nn.LayerNorm(C_t)
        self.token_mlp = nn.Sequential(
            nn.Linear(C_t, C_t),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(C_t, C_t),
        )
        self.token_dropout = nn.Dropout(0.2)
        self.ctx_enc    = ContextualROIEncoder(C_t=C_t, n_heads=n_heads, n_layers=n_layers)
        self.aggregator = AttentionAggregator(C_t=C_t)
        self.cls_head   = ClassificationHead(C_t=C_t, n_classes=n_classes)
        self.cbm        = ConceptBottleneck(K=K, C_t=C_t, n_classes=n_classes)

    # ------------------------------------------------------------------
    def forward(
        self,
        x: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> dict:
        """
        Full forward pass.

        Args:
            x         : (B, 1, H, W, D)   preprocessed MRI volume
            roi_masks : (K, h, w, d)       normalised binary ROI masks at
                                           feature-map resolution

        Returns dict with keys:
            F          : (B, C_f, h, w, d)  encoded feature map
            T          : (B, K, C_t)        ROI token matrix
            U          : (B, K, C_t)        contextualised tokens
            z          : (B, C_t)           global embedding
            alpha      : (B, K)             attention weights
            logits     : (B, C)             logits from z (direct classifier)
            c          : (B, K)             concept scores ∈ [0,1]
            cbm_logits : (B, C)             logits from concept bottleneck
        """
        F             = self.encoder(x)                      # (B, C_f, h, w, d)
        T             = self.tokenizer(F, roi_masks)         # (B, K, C_t)
        T             = self.token_norm(T)
        T             = T + self.token_mlp(T)
        T             = self.token_dropout(T)
        U             = self.ctx_enc(T)                     # (B, K, C_t)
        z, alpha      = self.aggregator(U)                   # (B, C_t), (B, K)
        logits        = self.cls_head(z)                     # (B, C)
        c, cbm_logits = self.cbm(U)                          # (B,K), (B,C)

        return {
            "F":          F,
            "T":          T,
            "U":          U,
            "z":          z,
            "alpha":      alpha,
            "logits":     logits,
            "c":          c,
            "cbm_logits": cbm_logits,
        }

    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict(
        self,
        x: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> dict:
        """
        Inference entry-point (Section P).

        Returns:
            y_hat  : (B,)        predicted class index
            p_tilde: (B, C)      probability from concept bottleneck
            c      : (B, K)      concept vector for explanation
            alpha  : (B, K)      ROI attention weights
        """
        out      = self.forward(x, roi_masks)
        p_tilde  = torch.softmax(out["cbm_logits"], dim=-1)
        y_hat    = p_tilde.argmax(dim=-1)
        return {
            "y_hat":   y_hat,
            "p_tilde": p_tilde,
            "c":       out["c"],
            "alpha":   out["alpha"],
        }


In [ ]:
"""
losses.py
=========
All training objectives from the M&M section (Sections H–N).

Loss map
--------
┌────────────────────────────┬──────────────────────────────────────────────┐
│ Symbol in M&M              │ Class / function here                        │
├────────────────────────────┼──────────────────────────────────────────────┤
│ L_cls                      │ ClassificationLoss           (Sec. H)        │
│ L_proto_align + L_proto_sep│ PrototypeLoss                (Sec. I)        │
│ L_pl                       │ PseudoLabelLoss              (Sec. M)        │
│ L_concept                  │ ConceptSupervisionLoss       (Sec. K)        │
│ L_anat                     │ AnatomicalConsistencyLoss    (Sec. L)        │
│ L_total                    │ TotalLoss                    (Sec. N)        │
└────────────────────────────┴──────────────────────────────────────────────┘

All loss modules are stateless (no internal buffers updated during forward);
prototype accumulators live in the trainer and are passed as arguments.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

class PredictionConsistencyLoss(nn.Module):
    """
    L_cons = KL( p_z || p_c )
    where p_z = softmax(logits_z) and p_c = softmax(logits_c).
    """

    def __init__(self):
        super().__init__()
        self.kl = nn.KLDivLoss(reduction="batchmean")

    def forward(self, logits_z: torch.Tensor, logits_c: torch.Tensor) -> torch.Tensor:
        log_p_z = F.log_softmax(logits_z, dim=-1)
        p_c = F.softmax(logits_c, dim=-1)
        return self.kl(log_p_z, p_c)


# ---------------------------------------------------------------------------
# 1.  Classification Loss  L_cls  (Section H)
# ---------------------------------------------------------------------------

class ClassificationLoss(nn.Module):
    """
    Supervised cross-entropy on labeled source mini-batch.

        L_cls = - Σ_{n ∈ B_S} Σ_c  y_{n,c} log p_{n,c}

    Uses logits + F.cross_entropy for numerical stability.

    Args:
        label_smoothing : float in [0, 1), optional label smoothing
    """

    def __init__(self, label_smoothing: float = 0.0):
        super().__init__()
        self.label_smoothing = label_smoothing

    def forward(
        self,
        logits: torch.Tensor,   # (B_S, C)
        labels: torch.Tensor,   # (B_S,)  integer class indices
    ) -> torch.Tensor:
        """
        Returns:
            scalar loss
        """
        return F.cross_entropy(
            logits, labels,
            label_smoothing=self.label_smoothing,
        )


# ---------------------------------------------------------------------------
# 2.  Prototype Loss  L_proto  (Section I)
# ---------------------------------------------------------------------------

class PrototypeLoss(nn.Module):
    """
    Class-conditional source–target prototype alignment + source separation.

        L_proto_align = Σ_c  ‖ μ_c^S − μ_c^T ‖²₂
        L_proto_sep   = Σ_{c≠c'} max(0, m − ‖ μ_c^S − μ_{c'}^S ‖₂)²
        L_proto       = L_proto_align + λ_sep · L_proto_sep

    Prototypes are computed **inside** this forward pass from the current
    mini-batch embeddings, using:
      - source labels  (hard)
      - target pseudo-labels only where confidence ≥ τ_p (Eq. I)

    Args:
        n_classes  : C
        tau_p      : confidence threshold for pseudo-labels
        margin     : separation margin m  (default 1.0)
        lambda_sep : weight of the separation term (default 0.1)
    """

    def __init__(
        self,
        n_classes: int,
        tau_p:      float = 0.9,
        margin:     float = 1.0,
        lambda_sep: float = 0.1,
    ):
        super().__init__()
        self.n_classes  = n_classes
        self.tau_p      = tau_p
        self.margin     = margin
        self.lambda_sep = lambda_sep

    # ------------------------------------------------------------------
    @staticmethod
    def _class_prototypes(
        z:       torch.Tensor,   # (B, C_t)
        labels:  torch.Tensor,   # (B,)  integer
        n_classes: int,
        eps: float = 1e-8,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Computes per-class mean embeddings.

        Returns:
            protos : (C, C_t)   — prototype per class (zero if class absent)
            valid  : (C,) bool  — True where prototype is non-trivial
        """
        C_t = z.shape[1]
        protos = z.new_zeros(n_classes, C_t)
        counts = z.new_zeros(n_classes)

        for c in range(n_classes):
            mask = labels == c
            if mask.any():
                protos[c] = z[mask].mean(dim=0)
                counts[c] = 1.0

        return protos, counts.bool()

    # ------------------------------------------------------------------
    def forward(
        self,
        z_src:      torch.Tensor,   # (B_S, C_t)  source embeddings
        y_src:      torch.Tensor,   # (B_S,)       source integer labels
        z_tgt:      torch.Tensor,   # (B_T, C_t)  target embeddings
        logits_tgt: torch.Tensor,   # (B_T, C)    target logits (for pseudo-label)
    ) -> tuple[torch.Tensor, dict]:
        """
        Returns:
            loss   : scalar
            info   : dict with 'align', 'sep', 'n_confident' for logging
        """
        # ---------- target pseudo-labels with confidence gate ----------
        probs_tgt = torch.softmax(logits_tgt, dim=-1)     # (B_T, C)
        conf, pseudo_labels = probs_tgt.max(dim=-1)       # (B_T,) each
        confident_mask = conf >= self.tau_p                # (B_T,)

        n_confident = int(confident_mask.sum().item())

        # ---------- source prototypes (always available) ---------------
        mu_src, valid_src = self._class_prototypes(
            z_src, y_src, self.n_classes
        )

        # ---------- alignment loss ------------------------------------
        align_loss = z_src.new_tensor(0.0)

        if n_confident > 0:
            z_conf  = z_tgt[confident_mask]
            pl_conf = pseudo_labels[confident_mask]

            mu_tgt, valid_tgt = self._class_prototypes(
                z_conf, pl_conf, self.n_classes
            )

            # align only for classes present in both source and target batch
            both_valid = valid_src & valid_tgt
            if both_valid.any():
                diff       = mu_src[both_valid] - mu_tgt[both_valid]  # (M, C_t)
                align_loss = (diff ** 2).sum(dim=-1).mean()

        # ---------- separation loss (source side only) ----------------
        sep_loss = z_src.new_tensor(0.0)
        valid_idx = valid_src.nonzero(as_tuple=True)[0]

        if len(valid_idx) >= 2:
            valid_protos = mu_src[valid_idx]                # (M, C_t)
            M = valid_protos.shape[0]
            sep_terms = []
            for i in range(M):
                for j in range(i + 1, M):
                    d = torch.norm(valid_protos[i] - valid_protos[j], p=2)
                    margin_violation = F.relu(self.margin - d) ** 2
                    sep_terms.append(margin_violation)

            if sep_terms:
                sep_loss = torch.stack(sep_terms).mean()

        # ---------- combined ------------------------------------------
        loss = align_loss + self.lambda_sep * sep_loss

        info = {
            "proto_align":   align_loss.item(),
            "proto_sep":     sep_loss.item(),
            "n_confident_T": n_confident,
        }
        return loss, info


# ---------------------------------------------------------------------------
# 3.  Pseudo-Label Self-Training Loss  L_pl  (Section M)
# ---------------------------------------------------------------------------

class PseudoLabelLoss(nn.Module):
    """
    Cross-entropy self-training on confident target samples.

        L_pl = - Σ_{m ∈ B^T_{τ_p}} Σ_c  ŷ_{m,c} log p_{m,c}

    where ŷ_{m,c} is the one-hot vector of the argmax pseudo-label.

    Args:
        tau_p : confidence threshold (same value as in PrototypeLoss)
    """

    def __init__(self, tau_p: float = 0.9):
        super().__init__()
        self.tau_p = tau_p

    def forward(
        self,
        logits_tgt: torch.Tensor,   # (B_T, C)
    ) -> tuple[torch.Tensor, int]:
        """
        Returns:
            loss        : scalar (0 if no confident sample)
            n_confident : int, for logging
        """
        probs      = torch.softmax(logits_tgt, dim=-1)
        conf, pseudo = probs.max(dim=-1)
        mask         = conf >= self.tau_p

        n_confident = int(mask.sum().item())

        if n_confident == 0:
            return logits_tgt.new_tensor(0.0), 0

        loss = F.cross_entropy(logits_tgt[mask], pseudo[mask])
        return loss, n_confident


# ---------------------------------------------------------------------------
# 4.  Concept Supervision Loss  L_concept  (Section K)
# ---------------------------------------------------------------------------

class ConceptSupervisionLoss(nn.Module):
    """
    Forces each concept score to match a pre-computed anatomical target.

        L_concept = Σ_{n ∈ B_S} Σ_k  | c_{n,k} − c̃_{n,k} |²

    where c̃_{n,k} is the normalised structural biomarker target (e.g.
    regional volume fraction, cortical thickness proxy).

    Targets are expected to be pre-normalised into [0, 1] to match the
    sigmoid activation of the concept head.
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        c: torch.Tensor,              # (B_S, K)  predicted concepts
        c_target: torch.Tensor,       # (B_S, K)  normalised anatomical targets
    ) -> torch.Tensor:
        """
        Returns:
            scalar MSE loss averaged over subjects and ROIs
        """
        return F.mse_loss(c, c_target)


# ---------------------------------------------------------------------------
# 5.  Anatomical Consistency Loss  L_anat  (Section L)
# ---------------------------------------------------------------------------

class AnatomicalConsistencyLoss(nn.Module):
    """
    Encourages concept scores to remain compatible with the Jacobian-based
    deformation summary (Section L).

        L_anat = Σ_n Σ_k  ω_k · | c_{n,k} − ḡ_{n,k} |²

    where ḡ_{n,k} = normalised -log(J) regional mean, and ω_k is an
    ROI-specific weight (default: uniform).

    Note: This loss acts as a plausibility regulariser, NOT a supervision
    signal claiming Jacobian determinants are ground-truth pathology.

    Args:
        K       : number of ROIs
        roi_weights : optional (K,) tensor of ω_k; uniform if None
    """

    def __init__(
        self,
        K: int,
        roi_weights: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        if roi_weights is None:
            roi_weights = torch.ones(K) / K
        # Register as buffer so it moves with .to(device)
        self.register_buffer("roi_weights", roi_weights)

    def forward(
        self,
        c: torch.Tensor,           # (B, K)  predicted concept scores
        g_bar: torch.Tensor,       # (B, K)  normalised deformation summaries
    ) -> torch.Tensor:
        """
        Returns:
            scalar weighted MSE loss
        """
        if g_bar.device != c.device:
            g_bar = g_bar.to(device=c.device, dtype=c.dtype, non_blocking=True)

        roi_weights = self.roi_weights
        if roi_weights.device != c.device or roi_weights.dtype != c.dtype:
            roi_weights = roi_weights.to(device=c.device, dtype=c.dtype, non_blocking=True)

        residuals = (c - g_bar) ** 2                      # (B, K)
        weighted  = residuals * roi_weights.unsqueeze(0) # (B, K)
        return weighted.mean()


# ---------------------------------------------------------------------------
# 6.  Combined Total Loss  L_total  (Section N)
# ---------------------------------------------------------------------------

# class SupervisedTotalLoss(nn.Module):
#     """
#     Supervised objective matching Sections I, J, K, L of the adjusted M&M.

#     Warm-up:
#         L_warm = lambda_z * L_cls^z

#     Full:
#         L_total = lambda_z    * L_cls^z
#                 + lambda_c    * L_cls^c
#                 + lambda_cons * L_cons
#                 + lambda_cbm  * L_concept
#                 + lambda_anat * L_anat
#     """

#     def __init__(
#         self,
#         n_classes: int,
#         K: int,
#         roi_weights: Optional[torch.Tensor] = None,
#         lambda_z: float = 1.0,
#         lambda_c: float = 1.0,
#         lambda_cons: float = 0.1,
#         lambda_cbm: float = 0.5,
#         lambda_anat: float = 0.2,
#         label_smoothing: float = 0.1,
#     ):
#         super().__init__()

#         self.lambda_z = float(lambda_z)
#         self.lambda_c = float(lambda_c)
#         self.lambda_cons = float(lambda_cons)
#         self.lambda_cbm = float(lambda_cbm)
#         self.lambda_anat = float(lambda_anat)

#         self.loss_cls_z = ClassificationLoss(label_smoothing=label_smoothing)
#         self.loss_cls_c = ClassificationLoss(label_smoothing=label_smoothing)
#         self.loss_cons = PredictionConsistencyLoss()
#         self.loss_concept = ConceptSupervisionLoss()
#         self.loss_anat = AnatomicalConsistencyLoss(K=K, roi_weights=roi_weights)

#     def forward_warm(
#         self,
#         logits_z: torch.Tensor,
#         labels: torch.Tensor,
#     ):
#         l_cls_z = self.loss_cls_z(logits_z, labels)

#         total = self.lambda_z * l_cls_z
#         info = {
#             "L_total": float(total.item()),
#             "L_cls_z": float(l_cls_z.item()),
#         }
#         return total, info

#     def forward_full(
#         self,
#         logits_z: torch.Tensor,
#         logits_c: torch.Tensor,
#         labels: torch.Tensor,
#         c: torch.Tensor,
#         c_target: torch.Tensor,
#         g_bar: torch.Tensor,
#     ):
#         l_cls_z = self.loss_cls_z(logits_z, labels)
#         l_cls_c = self.loss_cls_c(logits_c, labels)
#         l_cons = self.loss_cons(logits_z, logits_c)
#         l_concept = self.loss_concept(c, c_target)
#         l_anat = self.loss_anat(c, g_bar)

#         total = (
#             self.lambda_z    * l_cls_z
#             + self.lambda_c    * l_cls_c
#             + self.lambda_cons * l_cons
#             + self.lambda_cbm  * l_concept
#             + self.lambda_anat * l_anat
#         )

#         info = {
#             "L_total": float(total.item()),
#             "L_cls_z": float(l_cls_z.item()),
#             "L_cls_c": float(l_cls_c.item()),
#             "L_cons": float(l_cons.item()),
#             "L_concept": float(l_concept.item()),
#             "L_anat": float(l_anat.item()),
#         }
#         return total, info

from typing import Optional
import torch
import torch.nn as nn

import torch
import torch.nn as nn
from typing import Optional, Dict, Any


class DomainAdaptiveTotalLoss(nn.Module):
    """
    Domain-adaptive objective matching the PDF formulation:

        L_total =
            lambda_z    * L_cls^z
          + lambda_c    * L_cls^c
          + lambda_cons * L_cons
          + lambda_cbm  * L_concept
          + lambda_anat * L_anat
          + lambda_proto* L_proto
          + lambda_pl   * L_pl
          + lambda_wd   * ||Theta||^2

    Important:
    - Pseudo-labels for target are generated from the concept head logits
      (logits_c_tgt), following the PDF.
    - Prototype alignment is also driven by z_src, z_tgt and concept-head
      target pseudo-labels.
    """

    def __init__(
        self,
        n_classes: int,
        K: int,
        roi_weights: Optional[torch.Tensor] = None,
        # supervised weights
        lambda_z: float = 1.0,
        lambda_c: float = 1.0,
        lambda_cons: float = 0.1,
        lambda_cbm: float = 0.5,
        lambda_anat: float = 0.2,
        # adaptation weights
        lambda_proto: float = 0.2,
        lambda_pl: float = 0.1,
        # prototype config
        tau_p: float = 0.9,
        proto_margin: float = 1.0,
        lambda_sep: float = 0.1,
        # CE config
        label_smoothing: float = 0.1,
        # warm-up coefficients
        warm_lambda_z: float = 0.1,
        warm_lambda_c: float = 1.0,
        warm_lambda_cbm: float = 1.0,
        warm_lambda_anat: float = 1.0,
        warm_lambda_cons: float = 0.0,
    ):
        super().__init__()

        # -------- weights --------
        self.lambda_z = float(lambda_z)
        self.lambda_c = float(lambda_c)
        self.lambda_cons = float(lambda_cons)
        self.lambda_cbm = float(lambda_cbm)
        self.lambda_anat = float(lambda_anat)
        self.lambda_proto = float(lambda_proto)
        self.lambda_pl = float(lambda_pl)

        self.warm_lambda_z = float(warm_lambda_z)
        self.warm_lambda_c = float(warm_lambda_c)
        self.warm_lambda_cbm = float(warm_lambda_cbm)
        self.warm_lambda_anat = float(warm_lambda_anat)
        self.warm_lambda_cons = float(warm_lambda_cons)

        # -------- components already defined in your file --------
        self.loss_cls_z = ClassificationLoss(label_smoothing=label_smoothing)
        self.loss_cls_c = ClassificationLoss(label_smoothing=label_smoothing)
        self.loss_cons = PredictionConsistencyLoss()
        self.loss_concept = ConceptSupervisionLoss()
        self.loss_anat = AnatomicalConsistencyLoss(K=K, roi_weights=roi_weights)

        self.loss_proto = PrototypeLoss(
            n_classes=n_classes,
            tau_p=tau_p,
            margin=proto_margin,
            lambda_sep=lambda_sep,
        )

        self.loss_pl = PseudoLabelLoss(tau_p=tau_p)

    # ------------------------------------------------------------------
    # Stage I: source-only warm-up
    # ------------------------------------------------------------------
    def forward_warm(
        self,
        logits_z_src: torch.Tensor,   # (B_S, C)
        logits_c_src: torch.Tensor,   # (B_S, C)
        labels_src: torch.Tensor,     # (B_S,)
        c_src: torch.Tensor,          # (B_S, K)
        c_target_src: torch.Tensor,   # (B_S, K)
        g_bar_src: torch.Tensor,      # (B_S, K)
    ) -> tuple[torch.Tensor, Dict[str, Any]]:

        l_cls_z = self.loss_cls_z(logits_z_src, labels_src)
        l_cls_c = self.loss_cls_c(logits_c_src, labels_src)
        l_concept = self.loss_concept(c_src, c_target_src)
        l_anat = self.loss_anat(c_src, g_bar_src)
        l_cons = self.loss_cons(logits_z_src, logits_c_src)

        total = (
            self.warm_lambda_z    * self.lambda_z    * l_cls_z
            + self.warm_lambda_c    * self.lambda_c    * l_cls_c
            + self.warm_lambda_cbm  * self.lambda_cbm  * l_concept
            + self.warm_lambda_anat * self.lambda_anat * l_anat
            + self.warm_lambda_cons * self.lambda_cons * l_cons
        )

        info = {
            "L_total": float(total.item()),
            "L_cls_z": float(l_cls_z.item()),
            "L_cls_c": float(l_cls_c.item()),
            "L_cons": float(l_cons.item()),
            "L_concept": float(l_concept.item()),
            "L_anat": float(l_anat.item()),
            "L_proto": 0.0,
            "L_pl": 0.0,
            "proto_align": 0.0,
            "proto_sep": 0.0,
            "n_confident_T": 0,
        }
        return total, info

    # ------------------------------------------------------------------
    # Stage II: full domain adaptation
    # ------------------------------------------------------------------
    def forward_full(
        self,
        # ----- source -----
        logits_z_src: torch.Tensor,   # (B_S, C)
        logits_c_src: torch.Tensor,   # (B_S, C)
        labels_src: torch.Tensor,     # (B_S,)
        c_src: torch.Tensor,          # (B_S, K)
        c_target_src: torch.Tensor,   # (B_S, K)
        g_bar_src: torch.Tensor,      # (B_S, K)
        z_src: torch.Tensor,          # (B_S, C_t)
        # ----- target -----
        z_tgt: torch.Tensor,          # (B_T, C_t)
        logits_c_tgt: torch.Tensor,   # (B_T, C)  <-- concept-head for pseudo-labeling
    ) -> tuple[torch.Tensor, Dict[str, Any]]:

        # ----- supervised source losses -----
        l_cls_z = self.loss_cls_z(logits_z_src, labels_src)
        l_cls_c = self.loss_cls_c(logits_c_src, labels_src)
        l_cons = self.loss_cons(logits_z_src, logits_c_src)
        l_concept = self.loss_concept(c_src, c_target_src)
        l_anat = self.loss_anat(c_src, g_bar_src)

        # ----- domain adaptation losses -----
        l_proto, proto_info = self.loss_proto(
            z_src=z_src,
            y_src=labels_src,
            z_tgt=z_tgt,
            logits_tgt=logits_c_tgt,   # important: concept-head logits
        )

        l_pl, n_confident = self.loss_pl(
            logits_tgt=logits_c_tgt,   # important: concept-head logits
        )

        total = (
            self.lambda_z     * l_cls_z
            + self.lambda_c     * l_cls_c
            + self.lambda_cons  * l_cons
            + self.lambda_cbm   * l_concept
            + self.lambda_anat  * l_anat
            + self.lambda_proto * l_proto
            + self.lambda_pl    * l_pl
        )

        info = {
            "L_total": float(total.item()),
            "L_cls_z": float(l_cls_z.item()),
            "L_cls_c": float(l_cls_c.item()),
            "L_cons": float(l_cons.item()),
            "L_concept": float(l_concept.item()),
            "L_anat": float(l_anat.item()),
            "L_proto": float(l_proto.item()),
            "L_pl": float(l_pl.item()),
            "proto_align": float(proto_info["proto_align"]),
            "proto_sep": float(proto_info["proto_sep"]),
            "n_confident_T": int(n_confident),
        }
        return total, info


In [ ]:

"""
preprocessing.py
=================
Section B of the Materials and Methods.

This module standardizes a structural MRI volume into the common tensor space
expected by the model:
    X -> X_tilde -> X_bar

It is intentionally conservative. If the Kaggle derivatives are already
preprocessed, this module behaves as a consistency operator plus resampling
and intensity normalization.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Tuple, Union

import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F


ArrayLikePath = Union[str, Path]


@dataclass
class PreprocessConfig:
    target_shape: Tuple[int, int, int] = (128, 128, 128)
    eps: float = 1e-6
    brain_mask_threshold: float = 0.0
    clip_percentiles: Tuple[float, float] = (0.5, 99.5)
    enforce_canonical: bool = True


def load_nifti_canonical(path: ArrayLikePath, enforce_canonical: bool = True) -> tuple[np.ndarray, np.ndarray]:
    img = nib.load(str(path))
    if enforce_canonical:
        img = nib.as_closest_canonical(img)
    vol = img.get_fdata(dtype=np.float32)
    if vol.ndim == 4:
        vol = vol[..., 0]
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    return vol.astype(np.float32), img.affine.astype(np.float32)


def make_brain_mask(volume: np.ndarray, threshold: float = 0.0) -> np.ndarray:
    mask = np.isfinite(volume) & (volume > threshold)
    return mask.astype(np.float32)


def robust_clip_inside_mask(
    volume: np.ndarray,
    mask: np.ndarray,
    clip_percentiles: Tuple[float, float] = (0.5, 99.5),
) -> np.ndarray:
    vox = volume[mask > 0]
    if vox.size == 0:
        return volume.astype(np.float32)
    lo, hi = np.percentile(vox, clip_percentiles)
    return np.clip(volume, lo, hi).astype(np.float32)


def zscore_inside_mask(volume: np.ndarray, mask: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    vox = volume[mask > 0]
    if vox.size == 0:
        return volume.astype(np.float32)
    mu = float(vox.mean())
    sigma = float(vox.std())
    out = (volume - mu) / (sigma + eps)
    out[mask <= 0] = 0.0
    return out.astype(np.float32)


def resize_volume_torch(volume: np.ndarray, target_shape: Sequence[int]) -> np.ndarray:
    x = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0)  # (1,1,H,W,D)
    x = F.interpolate(x, size=tuple(target_shape), mode="trilinear", align_corners=False)
    return x.squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)


def preprocess_volume_array(volume: np.ndarray, cfg: PreprocessConfig) -> tuple[np.ndarray, np.ndarray]:
    mask = make_brain_mask(volume, threshold=cfg.brain_mask_threshold)
    volume = robust_clip_inside_mask(volume, mask, cfg.clip_percentiles)
    volume = resize_volume_torch(volume, cfg.target_shape)
    mask = resize_volume_torch(mask.astype(np.float32), cfg.target_shape)
    mask = (mask > 0.5).astype(np.float32)
    volume = zscore_inside_mask(volume, mask, eps=cfg.eps)
    return volume.astype(np.float32), mask.astype(np.float32)


def preprocess_nifti(
    path: ArrayLikePath,
    cfg: Optional[PreprocessConfig] = None,
    save_pt_path: Optional[ArrayLikePath] = None,
) -> dict:
    cfg = cfg or PreprocessConfig()
    volume, affine = load_nifti_canonical(path, enforce_canonical=cfg.enforce_canonical)
    x_bar, brain_mask = preprocess_volume_array(volume, cfg)

    tensor = torch.from_numpy(x_bar).unsqueeze(0)      # (1,H,W,D)
    mask_t = torch.from_numpy(brain_mask).unsqueeze(0) # (1,H,W,D)

    out = {
        "x": tensor.to(torch.float32),
        "brain_mask": mask_t.to(torch.float32),
        "affine": affine,
        "source_path": str(path),
    }

    if save_pt_path is not None:
        save_pt_path = Path(save_pt_path)
        save_pt_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(out, save_pt_path)

    return out


def validate_tensor_contract(sample: dict, expected_shape: Sequence[int] = (1, 128, 128, 128)) -> None:
    if "x" not in sample:
        raise KeyError("Missing key 'x' in preprocessed sample.")
    x = sample["x"]
    if not torch.is_tensor(x):
        raise TypeError("'x' must be a torch.Tensor.")
    if tuple(x.shape) != tuple(expected_shape):
        raise ValueError(f"Expected x shape {tuple(expected_shape)}, got {tuple(x.shape)}.")


In [ ]:

"""
dataset_contract.py
===================
Minimal dataset contract for the already-created dataloaders.

Required keys for Stage I source batches:
    x        : (B,1,H,W,D)
    y        : (B,)
    c_target : (B,K)
    g_bar    : (B,K)

Required keys for Stage II target batches:
    x        : (B,1,H,W,D)
    g_bar    : (B,K)

Optional metadata:
    subject_id, source_path, label_name
"""

from __future__ import annotations

from typing import Mapping

import torch


def validate_source_batch(batch: Mapping, K: int) -> None:
    for key in ["x", "y", "c_target", "g_bar"]:
        if key not in batch:
            raise KeyError(f"Missing key '{key}' in source batch.")
    if batch["x"].ndim != 5:
        raise ValueError(f"Expected x shape (B,1,H,W,D), got {tuple(batch['x'].shape)}.")
    if batch["c_target"].shape[-1] != K or batch["g_bar"].shape[-1] != K:
        raise ValueError("Concept targets or Jacobian summaries do not match K.")

def validate_target_batch(batch: Mapping, K: int) -> None:
    for key in ["x", "g_bar"]:
        if key not in batch:
            raise KeyError(f"Missing key '{key}' in target batch.")
    if batch["x"].ndim != 5:
        raise ValueError(f"Expected x shape (B,1,H,W,D), got {tuple(batch['x'].shape)}.")
    if batch["g_bar"].shape[-1] != K:
        raise ValueError("Jacobian summaries do not match K.")

In [ ]:
# ============================================================
# CEREBRA (.mnc) -> atlas discreto NIfTI listo para usar
# usando nibabel en lugar de SimpleITK
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_from_to

RAW_CEREBRA_PATH = "/kaggle/input/datasets/alejopatio/cerebra/mni_icbm152_CerebrA_tal_nlin_sym_09c.mnc"
OUT_DIR = "/kaggle/working/cerebra_prepared"
os.makedirs(OUT_DIR, exist_ok=True)

DISCRETE_ATLAS_PATH = os.path.join(OUT_DIR, "CerebrA_discrete_ready.nii.gz")
RESAMPLED_ATLAS_PATH = os.path.join(OUT_DIR, "CerebrA_discrete_resampled_to_reference.nii.gz")
LUT_CSV_PATH = os.path.join(OUT_DIR, "CerebrA_label_lut.csv")
META_JSON_PATH = os.path.join(OUT_DIR, "CerebrA_prepare_meta.json")


# ------------------------------------------------------------
# 1) UTILIDADES
# ------------------------------------------------------------
def read_image_any_nib(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"La ruta no existe: {path}")
    if not os.path.isfile(path):
        raise FileNotFoundError(f"La ruta existe pero no es un archivo: {path}")
    if os.path.getsize(path) == 0:
        raise RuntimeError(f"El archivo está vacío: {path}")

    try:
        img = nib.load(path)
        return img
    except Exception as e:
        raise RuntimeError(
            f"Nibabel no pudo abrir el archivo:\n{path}\n\n"
            f"Error original:\n{e}"
        )


def image_info_nib(img, path=""):
    hdr = img.header
    zooms = hdr.get_zooms()[:3] if len(hdr.get_zooms()) >= 3 else hdr.get_zooms()
    return {
        "path": path,
        "shape_xyz": tuple(int(v) for v in img.shape[:3]),
        "zooms_xyz": tuple(float(v) for v in zooms),
        "dtype": str(img.get_data_dtype()),
        "affine": np.asarray(img.affine).tolist(),
        "class": img.__class__.__name__,
    }


def unique_summary_nib(img, max_show=25):
    arr = np.asarray(img.dataobj)
    uniq = np.unique(arr)
    return {
        "shape": tuple(int(v) for v in arr.shape),
        "dtype": str(arr.dtype),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "n_unique": int(len(uniq)),
        "unique_head": uniq[:max_show].tolist(),
        "looks_integer": bool(np.allclose(uniq, np.round(uniq))),
    }


# ------------------------------------------------------------
# 2) PREPARAR CEREBRA COMO ATLAS DISCRETO
# ------------------------------------------------------------
def prepare_cerebra_discrete_atlas_nib(
    raw_atlas_path: str,
    out_atlas_path: str,
    lut_csv_path: str,
    meta_json_path: str,
    background_label: int = 0,
):
    img = read_image_any_nib(raw_atlas_path)
    img = nib.as_closest_canonical(img)

    info_before = image_info_nib(img, raw_atlas_path)
    uniq_before = unique_summary_nib(img)

    arr = np.asarray(img.dataobj)

    if arr.ndim != 3:
        raise ValueError(f"Se esperaba un atlas 3D, pero llegó shape {arr.shape}")

    # Discretización conservadora
    arr_disc = np.rint(arr).astype(np.int32)

    # Reindexar labels a 0,1,2,...,K
    unique_vals = np.unique(arr_disc)
    unique_vals = [int(v) for v in unique_vals if np.isfinite(v)]

    if background_label not in unique_vals:
        unique_vals = [background_label] + unique_vals

    roi_vals = sorted([v for v in unique_vals if v != background_label])

    old_to_new = {background_label: 0}
    for new_id, old_id in enumerate(roi_vals, start=1):
        old_to_new[old_id] = new_id

    remapped = np.zeros_like(arr_disc, dtype=np.int16)
    for old_id, new_id in old_to_new.items():
        remapped[arr_disc == old_id] = new_id

    out_img = nib.Nifti1Image(remapped, img.affine)
    out_img = nib.as_closest_canonical(out_img)
    nib.save(out_img, out_atlas_path)

    lut_df = pd.DataFrame({
        "old_label": list(old_to_new.keys()),
        "new_label": list(old_to_new.values()),
        "is_background": [int(k == background_label) for k in old_to_new.keys()]
    }).sort_values("new_label").reset_index(drop=True)
    lut_df.to_csv(lut_csv_path, index=False)

    info_after = image_info_nib(out_img, out_atlas_path)
    uniq_after = unique_summary_nib(out_img)

    meta = {
        "raw_atlas_path": raw_atlas_path,
        "discrete_atlas_path": out_atlas_path,
        "lut_csv_path": lut_csv_path,
        "background_label_old": int(background_label),
        "n_rois_excluding_background": int(lut_df["new_label"].max()),
        "image_info_before": info_before,
        "image_info_after": info_after,
        "unique_summary_before": uniq_before,
        "unique_summary_after": uniq_after,
    }

    with open(meta_json_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("\n--- ATLAS ORIGINAL ---")
    print(info_before)
    print("\nResumen de labels originales:")
    print(uniq_before)

    print("\n--- ATLAS DISCRETO ---")
    print(info_after)
    print("\nResumen de labels discretos:")
    print(uniq_after)

    print(f"\n[OK] Atlas discreto guardado en: {out_atlas_path}")
    print(f"[OK] LUT guardada en: {lut_csv_path}")
    print(f"[OK] Número de ROIs (sin fondo): {meta['n_rois_excluding_background']}")

    return meta

# ============================================================
# VERIFICACIÓN GEOMÉTRICA CON MRI DE REFERENCIA
# ============================================================

import glob
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to

def find_first_oasis_nifti(search_root="/kaggle/input"):
    candidates = sorted(
        glob.glob(os.path.join(search_root, "**", "*.nii"), recursive=True) +
        glob.glob(os.path.join(search_root, "**", "*.nii.gz"), recursive=True)
    )
    return candidates[0] if candidates else None


def compare_geometry_nib(img_a, img_b, atol_affine=1e-3, atol_zooms=1e-4):
    zooms_a = np.array(img_a.header.get_zooms()[:3], dtype=float)
    zooms_b = np.array(img_b.header.get_zooms()[:3], dtype=float)

    report = {
        "shape_a": tuple(int(v) for v in img_a.shape[:3]),
        "shape_b": tuple(int(v) for v in img_b.shape[:3]),
        "same_shape": bool(tuple(img_a.shape[:3]) == tuple(img_b.shape[:3])),
        "zooms_a": tuple(float(v) for v in zooms_a),
        "zooms_b": tuple(float(v) for v in zooms_b),
        "same_zooms": bool(np.allclose(zooms_a, zooms_b, atol=atol_zooms)),
        "same_affine": bool(np.allclose(img_a.affine, img_b.affine, atol=atol_affine)),
    }
    report["same_grid_exact"] = bool(
        report["same_shape"] and report["same_zooms"] and report["same_affine"]
    )
    return report


def resample_label_atlas_to_reference_nib(atlas_path, reference_mri_path, out_path):
    atlas_img = nib.load(atlas_path)
    ref_img = nib.load(reference_mri_path)

    # order=0 -> nearest neighbor para preservar labels
    atlas_resampled = resample_from_to(atlas_img, ref_img, order=0)
    atlas_resampled = nib.Nifti1Image(
        np.asarray(atlas_resampled.dataobj).astype(np.int16),
        atlas_resampled.affine
    )
    nib.save(atlas_resampled, out_path)
    return out_path

ATLAS_PATH = '/kaggle/input/datasets/alejopatio/cerebra/mni_icbm152_CerebrA_tal_nlin_sym_09c.mnc'

oasis_ref_path = find_first_oasis_nifti("/kaggle/input")

if oasis_ref_path is not None:
    print("MRI de referencia encontrado:")
    print(oasis_ref_path)

    atlas_img = nib.load(ATLAS_PATH)
    mri_img = nib.load(oasis_ref_path)

    report = compare_geometry_nib(atlas_img, mri_img)

    print("\n--- COMPARACIÓN GEOMÉTRICA ---")
    for k, v in report.items():
        print(f"{k}: {v}")

    if not report["same_grid_exact"]:
        print("\n[WARN] Atlas y MRI no comparten el mismo grid exacto.")
        print("       Voy a remuestrear el atlas al grid del MRI.")
        ATLAS_PATH = resample_label_atlas_to_reference_nib(
            atlas_path=ATLAS_PATH,
            reference_mri_path=oasis_ref_path,
            out_path=RESAMPLED_ATLAS_PATH
        )
        print("\nNuevo ATLAS_PATH:")
        print(ATLAS_PATH)
    else:
        print("\n[OK] Atlas y MRI comparten el mismo grid exacto.")
else:
    print("\n[WARN] No encontré un MRI NIfTI para verificación.")
    print("       Si solo tienes .pt, no puedes verificar affine/spacing retrospectivamente.")


meta = prepare_cerebra_discrete_atlas_nib(
    raw_atlas_path=RAW_CEREBRA_PATH,
    out_atlas_path=DISCRETE_ATLAS_PATH,
    lut_csv_path=LUT_CSV_PATH,
    meta_json_path=META_JSON_PATH,
    background_label=0,
)

ATLAS_PATH = DISCRETE_ATLAS_PATH
print("\nATLAS_PATH final:")
print(ATLAS_PATH)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import cycle
from pathlib import Path
from typing import Dict, Iterable, Optional, Any

import torch
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
)

@dataclass
class DomainAdaptiveTrainConfig:
    n_epochs_warm: int = 20
    n_epochs_full: int = 30
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    num_workers: int = 0
    device: str = "cpu"
    log_every: int = 10
    use_amp: bool = False


def _move_batch(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out


def _require_keys(batch: dict, keys: Iterable[str]) -> None:
    missing = [k for k in keys if k not in batch]
    if missing:
        raise KeyError(f"Batch is missing required keys: {missing}")

def _safe_macro_ovr_auc(y_true: np.ndarray, y_prob: np.ndarray, n_classes: int) -> float:
    aucs = []
    for c in range(n_classes):
        y_bin = (y_true == c).astype(np.int32)
        if y_bin.min() == y_bin.max():
            continue
        try:
            auc_c = roc_auc_score(y_bin, y_prob[:, c])
            aucs.append(float(auc_c))
        except Exception:
            continue

    if len(aucs) == 0:
        return float("nan")
    return float(np.mean(aucs))

def _classification_metrics_from_outputs(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    n_classes: int,
) -> dict:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro_ovr": _safe_macro_ovr_auc(y_true, y_prob, n_classes=n_classes),
    }


class DomainAdaptiveMRITrainer:
    def __init__(
        self,
        model,
        loss_fn,          # DomainAdaptiveTotalLoss
        atlas_mgr,
        input_shape=(128, 128, 128),
        cfg: Optional[DomainAdaptiveTrainConfig] = None,
        optimizer=None,
    ):
        self.model = model
        self.loss_fn = loss_fn
        self.atlas_mgr = atlas_mgr
        self.cfg = cfg or DomainAdaptiveTrainConfig()

        self.device = torch.device(self.cfg.device)
        self.model = self.model.to(self.device)
        self.loss_fn = self.loss_fn.to(self.device)

        if optimizer is None:
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.cfg.lr,
                weight_decay=self.cfg.weight_decay,
            )
        else:
            self.optimizer = optimizer

        self.scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.cfg.use_amp and self.device.type == "cuda"
        )

        self.roi_masks = self._build_feature_space_roi_masks(input_shape)

    def _build_feature_space_roi_masks(self, input_shape):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, *input_shape, device=self.device)
            feat = self.model.encoder(dummy)
            feature_shape = feat.shape[-3:]

        return self.atlas_mgr.get_masks(
            target_shape=feature_shape,
            normalize=True,
            device=self.device,
            dtype=torch.float32,
        )

    def _forward(self, x):
        return self.model(x, self.roi_masks)

    def _supervised_loss_info(self, batch, out, stage="full"):
        """
        Used only for evaluation on a single labeled loader.
        Domain adaptation losses (L_proto, L_pl) are NOT computed here because
        they require a source-target pair of batches.
        """
        _require_keys(batch, ["y", "c_target", "g_bar"])

        l_cls_z = self.loss_fn.loss_cls_z(out["logits"], batch["y"])
        l_cls_c = self.loss_fn.loss_cls_c(out["cbm_logits"], batch["y"])
        l_cons = self.loss_fn.loss_cons(out["logits"], out["cbm_logits"])
        l_concept = self.loss_fn.loss_concept(out["c"], batch["c_target"])
        l_anat = self.loss_fn.loss_anat(out["c"], batch["g_bar"])

        if stage == "warm":
            total = (
                self.loss_fn.warm_lambda_z    * self.loss_fn.lambda_z    * l_cls_z
                + self.loss_fn.warm_lambda_c    * self.loss_fn.lambda_c    * l_cls_c
                + self.loss_fn.warm_lambda_cbm  * self.loss_fn.lambda_cbm  * l_concept
                + self.loss_fn.warm_lambda_anat * self.loss_fn.lambda_anat * l_anat
                + self.loss_fn.warm_lambda_cons * self.loss_fn.lambda_cons * l_cons
            )
        elif stage == "full":
            total = (
                self.loss_fn.lambda_z    * l_cls_z
                + self.loss_fn.lambda_c    * l_cls_c
                + self.loss_fn.lambda_cons * l_cons
                + self.loss_fn.lambda_cbm  * l_concept
                + self.loss_fn.lambda_anat * l_anat
            )
        else:
            raise ValueError(f"stage must be 'warm' or 'full', got {stage!r}")

        return {
            "L_total": float(total.item()),
            "L_cls_z": float(l_cls_z.item()),
            "L_cls_c": float(l_cls_c.item()),
            "L_cons": float(l_cons.item()),
            "L_concept": float(l_concept.item()),
            "L_anat": float(l_anat.item()),
            "L_proto": 0.0,
            "L_pl": 0.0,
            "proto_align": 0.0,
            "proto_sep": 0.0,
            "n_confident_T": 0,
        }

    @torch.no_grad()
    def evaluate_supervised(self, loader, prefix="", stage="full"):
        """
        Evaluation on a single labeled loader.
        Used for:
            - source train
            - target train
            - source val
            - target val
        """
        self.model.eval()

        loss_sums = {}
        n_batches = 0

        y_true = []
        y_pred = []
        y_prob = []

        for batch in loader:
            batch = _move_batch(batch, self.device)
            out = self._forward(batch["x"])

            probs = torch.softmax(out["cbm_logits"], dim=-1)
            pred = probs.argmax(dim=-1)

            y_true.append(batch["y"].detach().cpu())
            y_pred.append(pred.detach().cpu())
            y_prob.append(probs.detach().cpu())

            info = self._supervised_loss_info(batch, out, stage=stage)

            for k, v in info.items():
                loss_sums[k] = loss_sums.get(k, 0.0) + float(v)

            n_batches += 1

        mean_losses = {k: v / max(n_batches, 1) for k, v in loss_sums.items()}

        y_true = torch.cat(y_true).numpy()
        y_pred = torch.cat(y_pred).numpy()
        y_prob = torch.cat(y_prob).numpy()

        metrics = _classification_metrics_from_outputs(
            y_true=y_true,
            y_pred=y_pred,
            y_prob=y_prob,
            n_classes=self.model.n_classes if hasattr(self.model, "n_classes") else y_prob.shape[1],
        )

        out_dict = {**mean_losses, **metrics}

        if prefix:
            out_dict = {f"{prefix}_{k}": v for k, v in out_dict.items()}

        return out_dict

    def train_warm_epoch(self, source_loader, epoch):
        self.model.train()

        meter = {}
        n_steps = 0

        for step, batch_src in enumerate(source_loader):
            _require_keys(batch_src, ["x", "y", "c_target", "g_bar"])
            batch_src = _move_batch(batch_src, self.device)

            self.optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=self.device.type,
                enabled=self.cfg.use_amp and self.device.type == "cuda",
            ):
                out_src = self._forward(batch_src["x"])

                loss, info = self.loss_fn.forward_warm(
                    logits_z_src=out_src["logits"],
                    logits_c_src=out_src["cbm_logits"],
                    labels_src=batch_src["y"],
                    c_src=out_src["c"],
                    c_target_src=batch_src["c_target"],
                    g_bar_src=batch_src["g_bar"],
                )

            if self.scaler.is_enabled():
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.optimizer.step()

            for k, v in info.items():
                meter[k] = meter.get(k, 0.0) + float(v)

            n_steps += 1

        meter = {k: v / max(n_steps, 1) for k, v in meter.items()}
        return meter

    def train_full_epoch(self, source_loader, target_loader, epoch):
        """
        Full domain adaptation epoch:
            - supervised source losses
            - prototype alignment
            - pseudo-label self-training on target
        """
        self.model.train()

        meter = {}
        n_steps = 0

        tgt_iter = cycle(target_loader)

        for step, batch_src in enumerate(source_loader):
            batch_tgt = next(tgt_iter)

            _require_keys(batch_src, ["x", "y", "c_target", "g_bar"])
            _require_keys(batch_tgt, ["x"])

            batch_src = _move_batch(batch_src, self.device)
            batch_tgt = _move_batch(batch_tgt, self.device)

            self.optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=self.device.type,
                enabled=self.cfg.use_amp and self.device.type == "cuda",
            ):
                out_src = self._forward(batch_src["x"])
                out_tgt = self._forward(batch_tgt["x"])

                loss, info = self.loss_fn.forward_full(
                    logits_z_src=out_src["logits"],
                    logits_c_src=out_src["cbm_logits"],
                    labels_src=batch_src["y"],
                    c_src=out_src["c"],
                    c_target_src=batch_src["c_target"],
                    g_bar_src=batch_src["g_bar"],
                    z_src=out_src["z"],
                    z_tgt=out_tgt["z"],
                    logits_c_tgt=out_tgt["cbm_logits"],  # pseudo-labels from concept head
                )

            if self.scaler.is_enabled():
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.optimizer.step()

            for k, v in info.items():
                meter[k] = meter.get(k, 0.0) + float(v)

            n_steps += 1

        meter = {k: v / max(n_steps, 1) for k, v in meter.items()}
        return meter

    @staticmethod
    def _fmt_metrics_block(info: dict, prefix: str) -> str:
        return (
            f"{prefix} "
            f"acc={info.get(f'{prefix}_accuracy', float('nan')):.4f} "
            f"f1={info.get(f'{prefix}_f1_macro', float('nan')):.4f} "
            f"prec={info.get(f'{prefix}_precision_macro', float('nan')):.4f} "
            f"rec={info.get(f'{prefix}_recall_macro', float('nan')):.4f} "
            f"auc={info.get(f'{prefix}_auc_macro_ovr', float('nan')):.4f}"
        )

    @staticmethod
    def _fmt_loss_block(info: dict) -> str:
        return (
            f"L_total={info.get('L_total', float('nan')):.4f} | "
            f"L_cls_z={info.get('L_cls_z', float('nan')):.4f} | "
            f"L_cls_c={info.get('L_cls_c', float('nan')):.4f} | "
            f"L_cons={info.get('L_cons', float('nan')):.4f} | "
            f"L_concept={info.get('L_concept', float('nan')):.4f} | "
            f"L_anat={info.get('L_anat', float('nan')):.4f} | "
            f"L_proto={info.get('L_proto', float('nan')):.4f} | "
            f"L_pl={info.get('L_pl', float('nan')):.4f} | "
            f"proto_align={info.get('proto_align', float('nan')):.4f} | "
            f"proto_sep={info.get('proto_sep', float('nan')):.4f} | "
            f"n_conf_T={int(info.get('n_confident_T', 0))}"
        )

    def fit(
        self,
        source_train_loader,
        target_train_loader,
        source_train_eval_loader=None,
        target_train_eval_loader=None,
        source_val_loader=None,
        target_val_loader=None,
    ):
        """
        Metrics are printed for:
            - train source
            - train target
            - val source
            - val target

        Loss block printed every epoch:
            - warm stage: supervised-only components
            - full stage: supervised + DA components
        """
        history = {"warm": [], "full": []}

        # -------------------------------------------------------------
        # Stage I: warm-up on source only
        # -------------------------------------------------------------
        for epoch in range(1, self.cfg.n_epochs_warm + 1):
            train_loss_info = self.train_warm_epoch(source_train_loader, epoch)

            tr_src = (
                self.evaluate_supervised(source_train_eval_loader, prefix="train_src", stage="warm")
                if source_train_eval_loader is not None else {}
            )
            tr_tgt = (
                self.evaluate_supervised(target_train_eval_loader, prefix="train_tgt", stage="warm")
                if target_train_eval_loader is not None else {}
            )
            va_src = (
                self.evaluate_supervised(source_val_loader, prefix="val_src", stage="warm")
                if source_val_loader is not None else {}
            )
            va_tgt = (
                self.evaluate_supervised(target_val_loader, prefix="val_tgt", stage="warm")
                if target_val_loader is not None else {}
            )

            epoch_info = {**train_loss_info, **tr_src, **tr_tgt, **va_src, **va_tgt}
            history["warm"].append(epoch_info)

            print(
                f"[Warm][Epoch {epoch:03d}] "
                f"{self._fmt_loss_block(epoch_info)} || "
                f"{self._fmt_metrics_block(epoch_info, 'train_src')} || "
                f"{self._fmt_metrics_block(epoch_info, 'train_tgt')} || "
                f"{self._fmt_metrics_block(epoch_info, 'val_src')} || "
                f"{self._fmt_metrics_block(epoch_info, 'val_tgt')}"
            )

        # -------------------------------------------------------------
        # Stage II: source-target adaptation
        # -------------------------------------------------------------
        for epoch in range(1, self.cfg.n_epochs_full + 1):
            train_loss_info = self.train_full_epoch(source_train_loader, target_train_loader, epoch)

            tr_src = (
                self.evaluate_supervised(source_train_eval_loader, prefix="train_src", stage="full")
                if source_train_eval_loader is not None else {}
            )
            tr_tgt = (
                self.evaluate_supervised(target_train_eval_loader, prefix="train_tgt", stage="full")
                if target_train_eval_loader is not None else {}
            )
            va_src = (
                self.evaluate_supervised(source_val_loader, prefix="val_src", stage="full")
                if source_val_loader is not None else {}
            )
            va_tgt = (
                self.evaluate_supervised(target_val_loader, prefix="val_tgt", stage="full")
                if target_val_loader is not None else {}
            )

            epoch_info = {**train_loss_info, **tr_src, **tr_tgt, **va_src, **va_tgt}
            history["full"].append(epoch_info)

            print(
                f"[Full][Epoch {epoch:03d}] "
                f"{self._fmt_loss_block(epoch_info)} || "
                f"{self._fmt_metrics_block(epoch_info, 'train_src')} || "
                f"{self._fmt_metrics_block(epoch_info, 'train_tgt')} || "
                f"{self._fmt_metrics_block(epoch_info, 'val_src')} || "
                f"{self._fmt_metrics_block(epoch_info, 'val_tgt')}"
            )

        return history




# Domain Adaptation training

In [ ]:
from sklearn.model_selection import train_test_split
from __future__ import annotations

import os
import sys
import glob
import json
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Optional, Tuple

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, roc_auc_score

# ---------------------------------------------------------------------
# 1) PATH BOOTSTRAP
# ---------------------------------------------------------------------
def add_module_dir_to_path(module_dir: str | os.PathLike) -> None:
    module_dir = str(module_dir)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)


def resolve_single_path(candidates) -> str:
    candidates = [str(p) for p in candidates if p and os.path.exists(str(p))]
    if not candidates:
        raise FileNotFoundError("No valid path found among candidates.")
    return candidates[0]


def discover_project_file(filename: str, search_roots: list[str]) -> str:
    hits = []
    for root in search_roots:
        if not root or not os.path.exists(root):
            continue
        hits.extend(glob.glob(os.path.join(root, "**", filename), recursive=True))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"Could not find {filename!r} in the provided roots.")
    return hits[0]

import torch
import torch.nn as nn

def pick_safe_device(verbose: bool = True) -> torch.device:
    """
    Selecciona CUDA solo si realmente es usable en esta sesión.
    Si CUDA existe pero la build actual de PyTorch no soporta la GPU asignada,
    hace fallback a CPU.
    """
    if not torch.cuda.is_available():
        if verbose:
            print("[Device] CUDA no disponible. Se usará CPU.")
        return torch.device("cpu")

    try:
        # prueba mínima de CUDA
        x = torch.zeros(1, device="cuda")
        _ = x + 1

        # prueba mínima de kernel real
        conv = nn.Conv3d(1, 2, kernel_size=3, padding=1).to("cuda")
        y = conv(torch.zeros(1, 1, 8, 8, 8, device="cuda"))
        _ = y.sum().item()

        if verbose:
            print(f"[Device] CUDA usable: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")

    except Exception as e:
        if verbose:
            print("[Device] CUDA detectada pero no usable con la build actual de PyTorch.")
            print(f"[Device] Fallback a CPU. Motivo: {type(e).__name__}: {e}")
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return torch.device("cpu")

# ---------------------------------------------------------------------
# 2) ROBUST .pt LOADING
# ---------------------------------------------------------------------
def load_tensor_like(obj_path: str) -> torch.Tensor:
    obj = torch.load(obj_path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict):
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj and torch.is_tensor(obj[key]):
                x = obj[key]
                break
        else:
            raise KeyError(f"Could not find tensor-like key in dict loaded from {obj_path}")
    elif torch.is_tensor(obj):
        x = obj
    else:
        raise TypeError(f"Unsupported object type loaded from {obj_path}: {type(obj)}")

    if x.ndim == 3:
        x = x.unsqueeze(0)  # (1,H,W,D)
    if x.ndim != 4:
        raise ValueError(f"Expected MRI tensor with shape (1,H,W,D), got {tuple(x.shape)} from {obj_path}")
    return x.to(torch.float32)


# ---------------------------------------------------------------------
# 3) METADATA RESOLUTION FROM THE USER'S BASE LAYOUT
# ---------------------------------------------------------------------
def build_source_path_map() -> Dict[str, str]:
    """
    Mirrors the user's base example: dynamically searches /kaggle/input for ADNI .pt
    tensors while excluding cached concept/Jacobian artifacts and OASIS files.
    """
    path_map = {}
    for f in glob.glob("/kaggle/input/**/*.pt", recursive=True):
        fl = f.lower()
        if any(tok in fl for tok in ["c_target", "g_bar", "g_jacobian", "target_oasis", "oasis"]):
            continue
        basename = os.path.basename(f).replace(".pt", "")
        path_map[basename] = f
    return path_map


def resolve_source_x_path(row: pd.Series, source_path_map: Dict[str, str]) -> str:
    sub_id = str(row["Subject_ID"])
    if sub_id in source_path_map:
        return source_path_map[sub_id]

    # fallback to whichever path-like columns may exist in the CSV
    for col in ["File_Path", "Raw_File_Path", "Processed_File_Path", "x_path"]:
        if col in row and pd.notna(row[col]) and os.path.exists(str(row[col])):
            return str(row[col])

    raise FileNotFoundError(f"Could not resolve source MRI path for Subject_ID={sub_id}")


def resolve_target_x_path(row: pd.Series, base_dir: str) -> str:
    sub_id = str(row["Subject_ID"])
    label = str(row["Label"])

    # First try the normalized storage layout produced in preprocessing.
    candidate = os.path.join(base_dir, "target_oasis", label, f"{sub_id}_MRI.pt")
    if os.path.exists(candidate):
        return candidate

    # Then trust any explicit path in the CSV if present.
    for col in ["Processed_File_Path", "File_Path", "Raw_File_Path", "x_path"]:
        if col in row and pd.notna(row[col]) and os.path.exists(str(row[col])):
            return str(row[col])

    raise FileNotFoundError(f"Could not resolve target MRI path for Subject_ID={sub_id}")


def build_inventory_dataframes(base_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    source_csv = os.path.join(base_dir, "source_labels.csv")
    target_csv = os.path.join(base_dir, "target_labels.csv")

    if not os.path.exists(source_csv):
        raise FileNotFoundError(f"Missing source CSV: {source_csv}")
    if not os.path.exists(target_csv):
        raise FileNotFoundError(f"Missing target CSV: {target_csv}")

    df_source = pd.read_csv(source_csv)
    df_target = pd.read_csv(target_csv)

    df_source = df_source[df_source["Label"].isin(LABEL_MAP)].reset_index(drop=True)
    df_target = df_target[df_target["Label"].isin(LABEL_MAP)].reset_index(drop=True)

    source_path_map = build_source_path_map()
    df_source = df_source.copy()
    df_target = df_target.copy()

    df_source["subject_id"] = df_source["Subject_ID"].astype(str)
    df_source["label"] = df_source["Label"].astype(str)
    df_source["x_path"] = df_source.apply(lambda r: resolve_source_x_path(r, source_path_map), axis=1)

    df_target["subject_id"] = df_target["Subject_ID"].astype(str)
    df_target["label"] = df_target["Label"].astype(str)
    df_target["x_path"] = df_target.apply(lambda r: resolve_target_x_path(r, base_dir), axis=1)

    return df_source[["subject_id", "label", "x_path"]], df_target[["subject_id", "label", "x_path"]]


        
class LabeledMRIDatasetWired(Dataset):
    """
    Dataset etiquetado para:
      - source train / source val
      - target train eval / target val
    Puede incluir c_target y g_bar.
    """
    def __init__(
        self,
        df_inventory: pd.DataFrame,
        df_jac: pd.DataFrame,
        K: int,
        df_concepts: Optional[pd.DataFrame] = None,
        require_concepts: bool = True,
    ):
        super().__init__()
        self.data = df_inventory.reset_index(drop=True)
        self.K = int(K)
        self.g_map = _index_by_subject(df_jac, "g_bar_path")
        self.c_map = None if df_concepts is None else _index_by_subject(df_concepts, "concept_target_path")
        self.require_concepts = bool(require_concepts)

        if self.require_concepts and self.c_map is None:
            raise ValueError("Este dataset requiere conceptos, pero df_concepts=None.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row["subject_id"])
        label_str = str(row["label"])

        x = load_tensor_like(str(row["x_path"]))
        y = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        g_bar = _load_vector(self.g_map[sub_id], expected_last_dim=self.K)

        item = {
            "x": x,
            "y": y,
            "g_bar": g_bar,
            "subject_id": sub_id,
            "label_name": label_str,
        }

        if self.c_map is not None:
            item["c_target"] = _load_vector(self.c_map[sub_id], expected_last_dim=self.K)

        return item


class UnlabeledTargetAdaptDataset(Dataset):
    """
    Dataset usado SOLO para adaptación target en Stage II.
    Devuelve únicamente lo necesario para forward target.
    """
    def __init__(self, df_inventory: pd.DataFrame):
        super().__init__()
        self.data = df_inventory.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row["subject_id"])

        x = load_tensor_like(str(row["x_path"]))
        return {
            "x": x,
            "subject_id": sub_id,
        }

def stratified_subject_split(
    df: pd.DataFrame,
    val_fraction: float = 0.2,
    random_state: int = 42,
):
    labels = df["label"].values
    idx = np.arange(len(df))

    train_idx, val_idx = train_test_split(
        idx,
        test_size=val_fraction,
        stratify=labels,
        random_state=random_state,
    )

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_val = df.iloc[val_idx].reset_index(drop=True)
    return df_train, df_val

def save_model_architecture(save_dir: str, model, cfg, extra_meta: Optional[dict] = None):
    os.makedirs(save_dir, exist_ok=True)

    arch_txt = os.path.join(save_dir, "model_architecture.txt")
    with open(arch_txt, "w", encoding="utf-8") as f:
        f.write(str(model))

    meta = {
        "trainer_cfg": asdict(cfg),
    }
    if extra_meta is not None:
        meta.update(extra_meta)

    meta_json = os.path.join(save_dir, "model_architecture_meta.json")
    with open(meta_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return {
        "architecture_txt": arch_txt,
        "architecture_meta_json": meta_json,
    }

# ---------------------------------------------------------------------
# 6) MODEL FACTORY WITH CONCEPT-HEAD PATCH
# ---------------------------------------------------------------------
def build_patched_model(
    project_root: str,
    K: int,
    n_classes: int = 3,
    C_f: int = 256,
    C_t: int = 128,
    n_heads: int = 4,
    n_layers: int = 2,
    base_ch: int = 32,
):
    add_module_dir_to_path(project_root)
    # from model import AlzheimerDomainAdaptationModel
    # from model_patch_concept import ConceptBottleneck as PatchedConceptBottleneck

    model = AlzheimerDomainAdaptationModel(
        K=K,
        C_f=C_f,
        C_t=C_t,
        n_classes=n_classes,
        n_heads=n_heads,
        n_layers=n_layers,
        base_ch=base_ch,
    )
    model.cbm = ConceptBottleneck(K=K, C_t=C_t, n_classes=n_classes)
    model.K = K
    return model
    
def train_domain_adaptation_fold(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    source_domain: str = "source",
    target_domain: str = "target",
    n_splits: int = 5,
    fold_idx: int = 0,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 10,
    n_epochs_full: int = 20,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    random_state: int = 42,
    save_dir: Optional[str] = None,
):
    source_domain = _normalize_domain_name(source_domain)
    target_domain = _normalize_domain_name(target_domain)

    if source_domain == target_domain:
        raise ValueError("source_domain y target_domain deben ser diferentes.")

    atlas_path = find_existing_atlas_path(atlas_path)
    add_module_dir_to_path(module_dir)
    add_module_dir_to_path(project_root)

    cache = load_precomputed_artifacts(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )

    K = int(cache["K"])

    src_tables = _get_domain_tables(cache, source_domain)
    tgt_tables = _get_domain_tables(cache, target_domain)

    df_source = src_tables["df_inventory"]
    df_source_concepts = src_tables["df_concepts"]
    df_source_jac = src_tables["df_jac"]

    df_target = tgt_tables["df_inventory"]
    df_target_concepts = tgt_tables["df_concepts"]
    df_target_jac = tgt_tables["df_jac"]

    # ------------------------------------------------------------
    # folds independientes en source y target
    # ------------------------------------------------------------
    source_splits = _make_stratified_splits(df_source, n_splits=n_splits, random_state=random_state)
    target_splits = _make_stratified_splits(df_target, n_splits=n_splits, random_state=random_state)

    if fold_idx < 0 or fold_idx >= n_splits:
        raise ValueError(f"fold_idx debe estar en [0, {n_splits-1}], pero llegó {fold_idx}")

    src_train_idx, src_val_idx = source_splits[fold_idx]
    tgt_train_idx, tgt_val_idx = target_splits[fold_idx]

    src_train_df = df_source.iloc[src_train_idx].reset_index(drop=True)
    src_val_df = df_source.iloc[src_val_idx].reset_index(drop=True)

    tgt_train_df = df_target.iloc[tgt_train_idx].reset_index(drop=True)
    tgt_val_df = df_target.iloc[tgt_val_idx].reset_index(drop=True)

    # ------------------------------------------------------------
    # datasets
    # ------------------------------------------------------------
    source_train_dataset = LabeledMRIDatasetWired(
        df_inventory=src_train_df,
        df_jac=df_source_jac,
        df_concepts=df_source_concepts,
        K=K,
        require_concepts=True,
    )

    source_val_dataset = LabeledMRIDatasetWired(
        df_inventory=src_val_df,
        df_jac=df_source_jac,
        df_concepts=df_source_concepts,
        K=K,
        require_concepts=True,
    )

    target_train_adapt_dataset = UnlabeledTargetAdaptDataset(
        df_inventory=tgt_train_df,
    )

    target_train_eval_dataset = LabeledMRIDatasetWired(
        df_inventory=tgt_train_df,
        df_jac=df_target_jac,
        df_concepts=df_target_concepts,
        K=K,
        require_concepts=True,
    )

    target_val_dataset = LabeledMRIDatasetWired(
        df_inventory=tgt_val_df,
        df_jac=df_target_jac,
        df_concepts=df_target_concepts,
        K=K,
        require_concepts=True,
    )

    use_pin_memory = torch.cuda.is_available()

    source_train_loader = DataLoader(
        source_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_train_loader = DataLoader(
        target_train_adapt_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    source_train_eval_loader = DataLoader(
        source_train_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_train_eval_loader = DataLoader(
        target_train_eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    source_val_loader = DataLoader(
        source_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_val_loader = DataLoader(
        target_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # ------------------------------------------------------------
    # atlas / model / loss / trainer
    # ------------------------------------------------------------
    atlas_mgr = AtlasROIManager(atlas_path)
    roi_weights = atlas_mgr.roi_weights_from_volume(power=0.0).to(torch.float32).cpu()

    model = build_patched_model(
        project_root=project_root,
        K=K,
        n_classes=len(LABEL_MAP),
    )

    loss_fn = DomainAdaptiveTotalLoss(
        n_classes=len(LABEL_MAP),
        K=K,
        roi_weights=roi_weights,
        lambda_z=1.0,
        lambda_c=1.0,
        lambda_cons=0.1,
        lambda_cbm=0.5,
        lambda_anat=0.2,
        lambda_proto=1.0,
        lambda_pl=0.1,
        tau_p=0.95,
        proto_margin=1.0,
        lambda_sep=0.1,
        label_smoothing=0.1,
        warm_lambda_z=0.1,
        warm_lambda_c=1.0,
        warm_lambda_cbm=1.0,
        warm_lambda_anat=1.0,
        warm_lambda_cons=0.0,
    )

    safe_device = pick_safe_device(verbose=True)

    cfg = DomainAdaptiveTrainConfig(
        n_epochs_warm=n_epochs_warm,
        n_epochs_full=n_epochs_full,
        lr=lr,
        weight_decay=weight_decay,
        num_workers=num_workers,
        device=safe_device.type,
        log_every=10,
        use_amp=(safe_device.type == "cuda"),
    )

    trainer = DomainAdaptiveMRITrainer(
        model=model,
        loss_fn=loss_fn,
        atlas_mgr=atlas_mgr,
        input_shape=(128, 128, 128),
        cfg=cfg,
    )

    print(f"[Fold {fold_idx+1}/{n_splits}] source={source_domain} target={target_domain}")
    print("trainer.device =", trainer.device)
    print("model device   =", next(trainer.model.parameters()).device)
    print("K              =", K)

    history = trainer.fit(
        source_train_loader=source_train_loader,
        target_train_loader=target_train_loader,
        source_train_eval_loader=source_train_eval_loader,
        target_train_eval_loader=target_train_eval_loader,
        source_val_loader=source_val_loader,
        target_val_loader=target_val_loader,
    )

    payload = {
        "fold_idx": fold_idx,
        "n_splits": n_splits,
        "source_domain": source_domain,
        "target_domain": target_domain,
        "history": history,
        "K": K,
        "atlas_path": atlas_path,
        "template_x_path": cache.get("template_x_path", None),
        "train_cfg": asdict(cfg),
        "n_source_train": len(source_train_dataset),
        "n_source_val": len(source_val_dataset),
        "n_target_train": len(target_train_adapt_dataset),
        "n_target_val": len(target_val_dataset),
    }

    if save_dir is not None:
        fold_dir = os.path.join(save_dir, f"{source_domain}_to_{target_domain}", f"fold_{fold_idx:02d}")
        os.makedirs(fold_dir, exist_ok=True)

        ckpt_path = os.path.join(fold_dir, "domain_adaptive_mri_cbm.pt")
        hist_path = os.path.join(fold_dir, "history_domain_adaptive.json")

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "K": K,
                "atlas_path": atlas_path,
                "train_cfg": asdict(cfg),
                "history": history,
                "fold_idx": fold_idx,
                "source_domain": source_domain,
                "target_domain": target_domain,
            },
            ckpt_path,
        )

        with open(hist_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        arch_paths = save_model_architecture(
            save_dir=fold_dir,
            model=model,
            cfg=cfg,
            extra_meta={
                "K": K,
                "atlas_path": atlas_path,
                "template_x_path": cache.get("template_x_path", None),
                "fold_idx": fold_idx,
                "source_domain": source_domain,
                "target_domain": target_domain,
            },
        )

        payload["checkpoint_path"] = ckpt_path
        payload["history_path"] = hist_path
        payload.update(arch_paths)

    return payload

def run_domain_adaptation_experiment(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    source_domain: str = "source",
    target_domain: str = "target",
    n_splits: int = 5,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 10,
    n_epochs_full: int = 20,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    random_state: int = 42,
    save_dir: Optional[str] = None,
):
    source_domain = _normalize_domain_name(source_domain)
    target_domain = _normalize_domain_name(target_domain)

    if source_domain == target_domain:
        raise ValueError("source_domain y target_domain deben ser diferentes.")

    all_results = []

    for fold_idx in range(n_splits):
        print("\n" + "=" * 100)
        print(
            f"Running domain adaptation fold {fold_idx + 1}/{n_splits} | "
            f"{source_domain} -> {target_domain}"
        )
        print("=" * 100)

        fold_result = train_domain_adaptation_fold(
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            source_domain=source_domain,
            target_domain=target_domain,
            n_splits=n_splits,
            fold_idx=fold_idx,
            batch_size=batch_size,
            num_workers=num_workers,
            n_epochs_warm=n_epochs_warm,
            n_epochs_full=n_epochs_full,
            lr=lr,
            weight_decay=weight_decay,
            random_state=random_state,
            save_dir=save_dir,
        )
        all_results.append(fold_result)

    return all_results

def run_bidirectional_domain_adaptation(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    n_splits: int = 5,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 10,
    n_epochs_full: int = 20,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    random_state: int = 42,
    save_dir: Optional[str] = None,
):
    forward_results = run_domain_adaptation_experiment(
        base_dir=base_dir,
        project_root=project_root,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
        source_domain="source",
        target_domain="target",
        n_splits=n_splits,
        batch_size=batch_size,
        num_workers=num_workers,
        n_epochs_warm=n_epochs_warm,
        n_epochs_full=n_epochs_full,
        lr=lr,
        weight_decay=weight_decay,
        random_state=random_state,
        save_dir=save_dir,
    )

    # backward_results = run_domain_adaptation_experiment(
    #     base_dir=base_dir,
    #     project_root=project_root,
    #     module_dir=module_dir,
    #     atlas_path=atlas_path,
    #     precomputed_artifacts_dir=precomputed_artifacts_dir,
    #     source_domain="target",
    #     target_domain="source",
    #     n_splits=n_splits,
    #     batch_size=batch_size,
    #     num_workers=num_workers,
    #     n_epochs_warm=n_epochs_warm,
    #     n_epochs_full=n_epochs_full,
    #     lr=lr,
    #     weight_decay=weight_decay,
    #     random_state=random_state,
    #     save_dir=save_dir,
    # )

    return {
        "source_to_target": forward_results,
        # "target_to_source": backward_results,
    }



from sklearn.model_selection import StratifiedKFold

def _normalize_domain_name(name: str) -> str:
    name = str(name).strip().lower()
    if name in {"source", "src", "adni"}:
        return "source"
    if name in {"target", "tgt", "oasis"}:
        return "target"
    raise ValueError(f"Dominio no reconocido: {name}")

def _get_domain_tables(cache: dict, domain_name: str):
    """
    Devuelve los dataframes correctos según el dominio elegido.
    source -> ADNI-like
    target -> OASIS-like
    """
    domain_name = _normalize_domain_name(domain_name)

    if domain_name == "source":
        return {
            "df_inventory": cache["df_source"].copy().reset_index(drop=True),
            "df_concepts": cache["df_concepts"].copy().reset_index(drop=True),
            "df_jac": cache["df_src_jac"].copy().reset_index(drop=True),
            "pretty_name": "source",
        }

    if domain_name == "target":
        if cache.get("df_tgt_concepts", None) is None:
            raise ValueError(
                "cache['df_tgt_concepts'] no existe. "
                "Para usar target como dominio evaluable o incluso como source en la dirección inversa, "
                "debes tener conceptos target precomputados."
            )
        return {
            "df_inventory": cache["df_target"].copy().reset_index(drop=True),
            "df_concepts": cache["df_tgt_concepts"].copy().reset_index(drop=True),
            "df_jac": cache["df_tgt_jac"].copy().reset_index(drop=True),
            "pretty_name": "target",
        }

    raise RuntimeError("No debería llegar aquí.")

def _make_stratified_splits(df: pd.DataFrame, n_splits: int, random_state: int = 42):
    """
    Construye folds estratificados usando la columna `label`.
    """
    y = df["label"].astype(str).values
    idx = np.arange(len(df))

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )
    return list(skf.split(idx, y))





In [ ]:


BASE_DIR = "/kaggle/input/notebooks/alejopatio/preprocess-alzheimer/model_ready_data"
PROJECT_ROOT = "/kaggle/input/notebooks/alejopatio/precompute-artifacts-alzheimer"   # donde están model.py y losses.py
MODULE_DIR = "/kaggle/working/mri_da_missing"
ATLAS_PATH = "/kaggle/input/notebooks/alejopatio/precompute-artifacts-alzheimer/cerebra_prepared/CerebrA_discrete_ready.nii.gz"

PRECOMP_DIR = "/kaggle/input/notebooks/alejopatio/precompute-artifacts-alzheimer/precomputed_artifacts_cerebra"
SAVE_DIR = "/kaggle/working/exp_da_cbm"

bidirectional_results = run_bidirectional_domain_adaptation(
    base_dir=BASE_DIR,
    project_root=PROJECT_ROOT,
    module_dir=MODULE_DIR,
    atlas_path=ATLAS_PATH,
    precomputed_artifacts_dir=PRECOMP_DIR,
    n_splits=5,
    batch_size=16,
    num_workers=2,
    n_epochs_warm=5,
    n_epochs_full=50,
    lr=1e-4,
    weight_decay=1e-4,
    random_state=42,
    save_dir=SAVE_DIR,
)

In [ ]:
bidirectional_results["source_to_target"]

# Ablation Study

In [ ]:
import os
import copy
import json
from typing import Optional, Dict, Any, List
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

class IdentityContextualEncoder(nn.Module):
    def forward(self, T):
        return T


class MeanPoolAggregator(nn.Module):
    """
    Sustituye la atención por promedio uniforme.
    Retorna:
        z     : (B, C_t)
        alpha : (B, K) uniforme
    """
    def forward(self, U):
        B, K, C_t = U.shape
        alpha = torch.full(
            (B, K),
            fill_value=1.0 / K,
            device=U.device,
            dtype=U.dtype,
        )
        z = U.mean(dim=1)
        return z, alpha





def apply_model_ablation(model, ablation_spec: Dict[str, Any]):
    patched = copy.deepcopy(model)
    patch_name = ablation_spec.get("model_patch", None)

    if patch_name is None:
        return patched

    if patch_name == "identity_ctx":
        patched.ctx_enc = IdentityContextualEncoder()
        return patched

    if patch_name == "mean_pool":
        patched.aggregator = MeanPoolAggregator()
        return patched

    raise ValueError(f"model_patch no reconocido: {patch_name}")


def build_ablation_loss(
    n_classes: int,
    K: int,
    roi_weights: torch.Tensor,
    ablation_spec: Dict[str, Any],
):
    base_kwargs = dict(
        n_classes=n_classes,
        K=K,
        roi_weights=roi_weights,
        lambda_z=1.0,
        lambda_c=1.0,
        lambda_cons=0.1,
        lambda_cbm=0.5,
        lambda_anat=0.2,
        lambda_proto=0.2,
        lambda_pl=0.1,
        tau_p=0.95,
        proto_margin=1.0,
        lambda_sep=0.1,
        label_smoothing=0.1,
        warm_lambda_z=0.1,
        warm_lambda_c=1.0,
        warm_lambda_cbm=1.0,
        warm_lambda_anat=1.0,
        warm_lambda_cons=0.0,
    )

    overrides = ablation_spec.get("loss_overrides", {})
    base_kwargs.update(overrides)

    return DomainAdaptiveTotalLoss(**base_kwargs)

def _get_last_epoch_info(history: dict) -> dict:
    if "full" in history and len(history["full"]) > 0:
        return history["full"][-1]
    if "warm" in history and len(history["warm"]) > 0:
        return history["warm"][-1]
    return {}


def extract_fold_terminal_metrics(run_payload: dict) -> dict:
    hist = run_payload["history"]
    last = _get_last_epoch_info(hist)

    row = {
        "ablation": run_payload.get("ablation_name", "unknown"),
        "fold_idx": run_payload.get("fold_idx", -1),
        "source_domain": run_payload.get("source_domain", None),
        "target_domain": run_payload.get("target_domain", None),

        # losses
        "L_total": last.get("L_total", np.nan),
        "L_cls_z": last.get("L_cls_z", np.nan),
        "L_cls_c": last.get("L_cls_c", np.nan),
        "L_cons": last.get("L_cons", np.nan),
        "L_concept": last.get("L_concept", np.nan),
        "L_anat": last.get("L_anat", np.nan),
        "L_proto": last.get("L_proto", np.nan),
        "L_pl": last.get("L_pl", np.nan),
        "proto_align": last.get("proto_align", np.nan),
        "proto_sep": last.get("proto_sep", np.nan),
        "n_confident_T": last.get("n_confident_T", 0),

        # source train
        "train_src_acc": last.get("train_src_accuracy", np.nan),
        "train_src_f1": last.get("train_src_f1_macro", np.nan),
        "train_src_precision": last.get("train_src_precision_macro", np.nan),
        "train_src_recall": last.get("train_src_recall_macro", np.nan),
        "train_src_auc": last.get("train_src_auc_macro_ovr", np.nan),

        # target train
        "train_tgt_acc": last.get("train_tgt_accuracy", np.nan),
        "train_tgt_f1": last.get("train_tgt_f1_macro", np.nan),
        "train_tgt_precision": last.get("train_tgt_precision_macro", np.nan),
        "train_tgt_recall": last.get("train_tgt_recall_macro", np.nan),
        "train_tgt_auc": last.get("train_tgt_auc_macro_ovr", np.nan),

        # source val
        "val_src_acc": last.get("val_src_accuracy", np.nan),
        "val_src_f1": last.get("val_src_f1_macro", np.nan),
        "val_src_precision": last.get("val_src_precision_macro", np.nan),
        "val_src_recall": last.get("val_src_recall_macro", np.nan),
        "val_src_auc": last.get("val_src_auc_macro_ovr", np.nan),

        # target val
        "val_tgt_acc": last.get("val_tgt_accuracy", np.nan),
        "val_tgt_f1": last.get("val_tgt_f1_macro", np.nan),
        "val_tgt_precision": last.get("val_tgt_precision_macro", np.nan),
        "val_tgt_recall": last.get("val_tgt_recall_macro", np.nan),
        "val_tgt_auc": last.get("val_tgt_auc_macro_ovr", np.nan),
    }
    return row

def summarize_ablation_results(df_folds: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [c for c in df_folds.columns if c not in {"ablation", "fold_idx", "source_domain", "target_domain"}]

    rows = []
    for ablation_name, g in df_folds.groupby("ablation"):
        row = {"ablation": ablation_name}
        for col in metric_cols:
            vals = pd.to_numeric(g[col], errors="coerce").values.astype(float)
            row[f"{col}_mean"] = float(np.nanmean(vals))
            row[f"{col}_std"] = float(np.nanstd(vals))
        rows.append(row)

    return pd.DataFrame(rows).sort_values("ablation").reset_index(drop=True)

def train_domain_adaptation_fold(
        base_dir: str,
        project_root: str,
        module_dir: str,
        atlas_path: str,
        precomputed_artifacts_dir: str,
        source_domain: str = "source",
        target_domain: str = "target",
        n_splits: int = 5,
        fold_idx: int = 0,
        batch_size: int = 2,
        num_workers: int = 0,
        n_epochs_warm: int = 10,
        n_epochs_full: int = 20,
        lr: float = 1e-4,
        weight_decay: float = 1e-4,
        random_state: int = 42,
        save_dir: Optional[str] = None,
        ablation_spec: Optional[Dict[str, Any]] = None,
    ):


    ablation_spec = copy.deepcopy(ablation_spec or {
        "name": "full",
        "description": "Modelo completo",
        "loss_overrides": {},
        "model_patch": None,
    })
    
    source_domain = _normalize_domain_name(source_domain)
    target_domain = _normalize_domain_name(target_domain)

    if source_domain == target_domain:
        raise ValueError("source_domain y target_domain deben ser diferentes.")

    atlas_path = find_existing_atlas_path(atlas_path)
    add_module_dir_to_path(module_dir)
    add_module_dir_to_path(project_root)

    cache = load_precomputed_artifacts(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )

    K = int(cache["K"])

    src_tables = _get_domain_tables(cache, source_domain)
    tgt_tables = _get_domain_tables(cache, target_domain)

    df_source = src_tables["df_inventory"]
    df_source_concepts = src_tables["df_concepts"]
    df_source_jac = src_tables["df_jac"]

    df_target = tgt_tables["df_inventory"]
    df_target_concepts = tgt_tables["df_concepts"]
    df_target_jac = tgt_tables["df_jac"]

    # ------------------------------------------------------------
    # folds independientes en source y target
    # ------------------------------------------------------------
    source_splits = _make_stratified_splits(df_source, n_splits=n_splits, random_state=random_state)
    target_splits = _make_stratified_splits(df_target, n_splits=n_splits, random_state=random_state)

    if fold_idx < 0 or fold_idx >= n_splits:
        raise ValueError(f"fold_idx debe estar en [0, {n_splits-1}], pero llegó {fold_idx}")

    src_train_idx, src_val_idx = source_splits[fold_idx]
    tgt_train_idx, tgt_val_idx = target_splits[fold_idx]

    src_train_df = df_source.iloc[src_train_idx].reset_index(drop=True)
    src_val_df = df_source.iloc[src_val_idx].reset_index(drop=True)

    tgt_train_df = df_target.iloc[tgt_train_idx].reset_index(drop=True)
    tgt_val_df = df_target.iloc[tgt_val_idx].reset_index(drop=True)

    # ------------------------------------------------------------
    # datasets
    # ------------------------------------------------------------
    source_train_dataset = LabeledMRIDatasetWired(
        df_inventory=src_train_df,
        df_jac=df_source_jac,
        df_concepts=df_source_concepts,
        K=K,
        require_concepts=True,
    )

    source_val_dataset = LabeledMRIDatasetWired(
        df_inventory=src_val_df,
        df_jac=df_source_jac,
        df_concepts=df_source_concepts,
        K=K,
        require_concepts=True,
    )

    target_train_adapt_dataset = UnlabeledTargetAdaptDataset(
        df_inventory=tgt_train_df,
    )

    target_train_eval_dataset = LabeledMRIDatasetWired(
        df_inventory=tgt_train_df,
        df_jac=df_target_jac,
        df_concepts=df_target_concepts,
        K=K,
        require_concepts=True,
    )

    target_val_dataset = LabeledMRIDatasetWired(
        df_inventory=tgt_val_df,
        df_jac=df_target_jac,
        df_concepts=df_target_concepts,
        K=K,
        require_concepts=True,
    )

    use_pin_memory = torch.cuda.is_available()

    source_train_loader = DataLoader(
        source_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_train_loader = DataLoader(
        target_train_adapt_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    source_train_eval_loader = DataLoader(
        source_train_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_train_eval_loader = DataLoader(
        target_train_eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    source_val_loader = DataLoader(
        source_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_val_loader = DataLoader(
        target_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # ------------------------------------------------------------
    # atlas / model / loss / trainer
    # ------------------------------------------------------------
    atlas_mgr = AtlasROIManager(atlas_path)
    roi_weights = atlas_mgr.roi_weights_from_volume(power=0.0).to(torch.float32).cpu()
    
    base_model = build_patched_model(
        project_root=project_root,
        # module_dir=module_dir,
        K=K,
        n_classes=len(LABEL_MAP),
    )
    
    model = apply_model_ablation(base_model, ablation_spec)
    
    loss_fn = build_ablation_loss(
        n_classes=len(LABEL_MAP),
        K=K,
        roi_weights=roi_weights,
        ablation_spec=ablation_spec,
    )

    safe_device = pick_safe_device(verbose=True)

    cfg = DomainAdaptiveTrainConfig(
        n_epochs_warm=n_epochs_warm,
        n_epochs_full=n_epochs_full,
        lr=lr,
        weight_decay=weight_decay,
        num_workers=num_workers,
        device=safe_device.type,
        log_every=10,
        use_amp=(safe_device.type == "cuda"),
    )

    trainer = DomainAdaptiveMRITrainer(
        model=model,
        loss_fn=loss_fn,
        atlas_mgr=atlas_mgr,
        input_shape=(128, 128, 128),
        cfg=cfg,
    )

    print(f"[Fold {fold_idx+1}/{n_splits}] source={source_domain} target={target_domain}")
    print("trainer.device =", trainer.device)
    print("model device   =", next(trainer.model.parameters()).device)
    print("K              =", K)

    history = trainer.fit(
        source_train_loader=source_train_loader,
        target_train_loader=target_train_loader,
        source_train_eval_loader=source_train_eval_loader,
        target_train_eval_loader=target_train_eval_loader,
        source_val_loader=source_val_loader,
        target_val_loader=target_val_loader,
    )

    payload = {
        "fold_idx": fold_idx,
        "n_splits": n_splits,
        "source_domain": source_domain,
        "target_domain": target_domain,
        "history": history,
        "K": K,
        "atlas_path": atlas_path,
        "template_x_path": cache.get("template_x_path", None),
        "train_cfg": asdict(cfg),
        "n_source_train": len(source_train_dataset),
        "n_source_val": len(source_val_dataset),
        "n_target_train": len(target_train_adapt_dataset),
        "n_target_val": len(target_val_dataset),
    }
    payload["ablation_name"] = ablation_spec["name"]
    payload["ablation_description"] = ablation_spec["description"]

    if save_dir is not None:
        fold_dir = os.path.join(
            save_dir,
            ablation_spec["name"],
            f"{source_domain}_to_{target_domain}",
            f"fold_{fold_idx:02d}"
        )
        os.makedirs(fold_dir, exist_ok=True)

        ckpt_path = os.path.join(fold_dir, "domain_adaptive_mri_cbm.pt")
        hist_path = os.path.join(fold_dir, "history_domain_adaptive.json")

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "K": K,
                "atlas_path": atlas_path,
                "train_cfg": asdict(cfg),
                "history": history,
                "fold_idx": fold_idx,
                "source_domain": source_domain,
                "target_domain": target_domain,
            },
            ckpt_path,
        )

        with open(hist_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        arch_paths = save_model_architecture(
            save_dir=fold_dir,
            model=model,
            cfg=cfg,
            extra_meta={
                "K": K,
                "atlas_path": atlas_path,
                "template_x_path": cache.get("template_x_path", None),
                "fold_idx": fold_idx,
                "source_domain": source_domain,
                "target_domain": target_domain,
            },
        )

        payload["checkpoint_path"] = ckpt_path
        payload["history_path"] = hist_path
        
        payload.update(arch_paths)

    return payload

def run_domain_adaptation_experiment(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    source_domain: str = "source",
    target_domain: str = "target",
    n_splits: int = 5,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 10,
    n_epochs_full: int = 20,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    random_state: int = 42,
    save_dir: Optional[str] = None,
    ablation_spec: Optional[Dict[str, Any]] = None,
):
    source_domain = _normalize_domain_name(source_domain)
    target_domain = _normalize_domain_name(target_domain)

    if source_domain == target_domain:
        raise ValueError("source_domain y target_domain deben ser diferentes.")

    all_results = []

    for fold_idx in range(n_splits):
        availability = False 
        if ( ablation_spec["name"] == "no_ctx_encoder" and fold_idx == 4):
            availability = True 

        elif ( ablation_spec["name"] == "no_concept" and fold_idx >= 3):
            availability = True 

        elif ( ablation_spec["name"] == "no_pl" and fold_idx == 4):
            availability = True 

        elif ( ablation_spec["name"] == "no_domain_adaptation" and fold_idx >= 3):
            availability = True 

        if availability:
            print("\n" + "=" * 100)
            print(
                f"Running domain adaptation fold {fold_idx + 1}/{n_splits} | "
                f"{source_domain} -> {target_domain}"
            )
            print("=" * 100)
    
            fold_result = train_domain_adaptation_fold(
                base_dir=base_dir,
                project_root=project_root,
                module_dir=module_dir,
                atlas_path=atlas_path,
                precomputed_artifacts_dir=precomputed_artifacts_dir,
                source_domain=source_domain,
                target_domain=target_domain,
                n_splits=n_splits,
                fold_idx=fold_idx,
                batch_size=batch_size,
                num_workers=num_workers,
                n_epochs_warm=n_epochs_warm,
                n_epochs_full=n_epochs_full,
                lr=lr,
                weight_decay=weight_decay,
                random_state=random_state,
                save_dir=save_dir,
                ablation_spec=ablation_spec,
            )
            all_results.append(fold_result)

    return all_results

def run_ablation_study(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    source_domain: str = "source",
    target_domain: str = "target",
    n_splits: int = 3,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 5,
    n_epochs_full: int = 10,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    random_state: int = 42,
    save_dir: Optional[str] = None,
    ablation_specs: Optional[List[Dict[str, Any]]] = None,
):
    """
    Recomendación práctica:
    - usar n_splits=3 y pocas épocas para screening
    - luego repetir solo las mejores 2-3 con n_splits=5
    """
    if ablation_specs is None:
        ablation_specs = get_default_ablation_specs(include_no_da=False)

    all_runs = []
    fold_rows = []

    for spec in ablation_specs:
        print("\n" + "#" * 120)
        print(f"ABLATION: {spec['name']} | {spec['description']}")
        print("#" * 120)

        spec_save_dir = None if save_dir is None else os.path.join(save_dir, spec["name"])

        run_results = run_domain_adaptation_experiment(
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            source_domain=source_domain,
            target_domain=target_domain,
            n_splits=n_splits,
            batch_size=batch_size,
            num_workers=num_workers,
            n_epochs_warm=n_epochs_warm,
            n_epochs_full=n_epochs_full,
            lr=lr,
            weight_decay=weight_decay,
            random_state=random_state,
            save_dir=spec_save_dir,
            ablation_spec=spec,
        )

        all_runs.append({
            "ablation": spec["name"],
            "description": spec["description"],
            "folds": run_results,
        })

        for fr in run_results:
            row = extract_fold_terminal_metrics(fr)
            fold_rows.append(row)

        current_df = pd.DataFrame(fold_rows)
        print("\n[Partial fold-level results]")
        print(current_df.to_string(index=False))

        current_summary = summarize_ablation_results(current_df)
        print("\n[Partial summary mean ± std]")
        print(current_summary.to_string(index=False))

    folds_df = pd.DataFrame(fold_rows)
    summary_df = summarize_ablation_results(folds_df)

    print("\n" + "=" * 120)
    print("FINAL FOLD-LEVEL RESULTS")
    print("=" * 120)
    print(folds_df.to_string(index=False))

    print("\n" + "=" * 120)
    print("FINAL SUMMARY (mean ± std)")
    print("=" * 120)
    print(summary_df.to_string(index=False))

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

        folds_csv = os.path.join(save_dir, "ablation_folds.csv")
        summary_csv = os.path.join(save_dir, "ablation_summary.csv")
        runs_json = os.path.join(save_dir, "ablation_runs.json")

        folds_df.to_csv(folds_csv, index=False)
        summary_df.to_csv(summary_csv, index=False)

        with open(runs_json, "w", encoding="utf-8") as f:
            json.dump(all_runs, f, indent=2)

        print(f"\nSaved fold-level results to: {folds_csv}")
        print(f"Saved summary results to:    {summary_csv}")
        print(f"Saved run metadata to:       {runs_json}")

    return {
        "all_runs": all_runs,
        "folds_df": folds_df,
        "summary_df": summary_df,
    }

In [ ]:
def get_default_ablation_specs(include_no_da: bool = False) -> List[Dict[str, Any]]:
    specs = [
        {
            "name": "full",
            "description": "Modelo completo",
            "loss_overrides": {},
            "model_patch": None,
        },
        {
            "name": "no_proto",
            "description": "Sin prototype alignment",
            "loss_overrides": {"lambda_proto": 0.0},
            "model_patch": None,
        },
        {
            "name": "no_pl",
            "description": "Sin pseudo-label self-training",
            "loss_overrides": {"lambda_pl": 0.0},
            "model_patch": None,
        },
        {
            "name": "no_cons",
            "description": "Sin consistency loss entre cabezas",
            "loss_overrides": {"lambda_cons": 0.0},
            "model_patch": None,
        },
        {
            "name": "no_concept",
            "description": "Sin concept supervision",
            "loss_overrides": {"lambda_cbm": 0.0},
            "model_patch": None,
        },
        {
            "name": "no_anat",
            "description": "Sin anatomical consistency",
            "loss_overrides": {"lambda_anat": 0.0},
            "model_patch": None,
        },
        {
            "name": "no_ctx_encoder",
            "description": "Sin Transformer contextual ROI encoder",
            "loss_overrides": {},
            "model_patch": "identity_ctx",
        },
        {
            "name": "mean_pool",
            "description": "Sin attention aggregation; mean pooling uniforme",
            "loss_overrides": {},
            "model_patch": "mean_pool",
        },
    ]

    if include_no_da:
        specs.append(
            {
                "name": "no_domain_adaptation",
                "description": "Sin prototype alignment ni pseudo-labeling",
                "loss_overrides": {
                    "lambda_proto": 0.0,
                    "lambda_pl": 0.0,
                },
                "model_patch": None,
            }
        )

    return specs

# ABLATIONS_FAST = get_default_ablation_specs(True)

# ablation_results = run_ablation_study(
#     base_dir=BASE_DIR,
#     project_root=PROJECT_ROOT,
#     module_dir=MODULE_DIR,
#     atlas_path=ATLAS_PATH,
#     precomputed_artifacts_dir=PRECOMP_DIR,
#     source_domain="source",
#     target_domain="target",
#     n_splits=5,          # screening rápido
#     batch_size=16,
#     num_workers=2,
#     n_epochs_warm=5,
#     n_epochs_full=50,
#     lr=1e-4,
#     weight_decay=1e-4,
#     random_state=42,
#     save_dir="/kaggle/working/ablation_da_cbm",
#     ablation_specs=ABLATIONS_FAST,
# )